<!-- # C1 回路による量子乱数生成 -->

# Quantum Random Number Generation with the C1 Circuit

<!-- ## このノートブックで行うこと -->

## What this notebook does

<!-- 1 量子ビットだけを使う、最も簡単な量子乱数生成回路（**C1 回路**）を Qiskit で実装し、得られたビット列の偏りとエントロピーを調べます。流れは次のとおりです。 -->

We implement the **C1 circuit**, the simplest quantum random number generation circuit, which uses only one qubit, in Qiskit and examine the bias and entropy of the resulting bit string. The workflow is as follows.

<!--
1. C1 回路を作る（重ね合わせを作るゲートを Hadamard ゲート $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$ の中から 1 つ選ぶ）
2. 回路を量子コンピュータのネイティブゲートに変換（transpile）する
3. **ノイズなし**のシミュレータで測定し、評価指標を計算する
4. 実機相当の**ノイズあり**のシミュレータで測定し、評価指標を計算する
5. 同じ測定を 10 回繰り返し、結果のばらつきを見る
6. ノイズの強さを段階的に変え、結果への影響を見る（5.9）
-->

1. Build the C1 circuit (choose one gate to create the superposition from the Hadamard gate $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, and $R_Y(\pi/2)$)
2. Convert (transpile) the circuit into the native gates of the quantum computer
3. Measure on a **noiseless** simulator and compute the evaluation metrics
4. Measure on a **noisy** simulator that mimics real hardware and compute the evaluation metrics
5. Repeat the same measurement 10 times to see how much the results vary
6. Change the noise strength step by step to see its effect on the results (5.9)

<!-- この説明パート（A〜G）で理論と使う道具をひととおり説明し、後半の各節（1.〜5.）でその内容をコードと並べて一つずつ確かめていきます。 -->

This explanatory part (A–G) goes through the theory and the tools used, and the later sections (1.–5.) then check that material one step at a time alongside the code.

---

<!-- ### A. 先行研究と C1 回路 -->

### A. Prior work and the C1 circuit

<!-- このノートブックの回路と、ノイズのパラメータの数値は、次の論文にもとづいています。ノイズモデルの組み立て方は論文のものではなく、このノートブックで選んだものです（E を参照）。 -->

The circuit and the numerical noise parameters in this notebook are based on the following paper. How the noise model is assembled is not taken from the paper; it was chosen for this notebook (see E).

> A. Gnatowski *et al.*, "True Random Number Generators on IQM Spark," arXiv:2512.09862 (2025).

<!-- この論文は、Wrocław University of Science and Technology に設置された 5 量子ビットの超伝導量子コンピュータ **Odra 5**（IQM Spark）で、5 種類の乱数生成回路 C1〜C5 を比較しています。C1 回路は論文で次のように説明されています。 -->

This paper compares five random number generation circuits, C1–C5, on **Odra 5** (IQM Spark), a 5-qubit superconducting quantum computer installed at Wrocław University of Science and Technology. The paper describes the C1 circuit as follows.

> "initializing a qubit in the $|0\rangle$ state, applying a single quantum gate—either a Hadamard ($H$) gate, a rotation gate $R_x(\pi/2)$, or $R_y(\pi/2)$—and measuring the qubit in the computational basis"

<!-- つまり、$|0\rangle$ に 1 個のゲートを作用させて測定するだけの回路です。 -->

In other words, the circuit simply applies one gate to $|0\rangle$ and measures.

<!-- このノートブックでは、Shannon エントロピー $H_{\mathrm{Shannon}}$（C を参照）と区別するため、Hadamard ゲートを $H_{\mathrm{Gate}}$ と書きます。引用文中の $H$ と同じゲートです。コード内のゲート名やグラフの凡例は `"H"` のままです。 -->

In this notebook, the Hadamard gate is written $H_{\mathrm{Gate}}$ to distinguish it from the Shannon entropy $H_{\mathrm{Shannon}}$ (see C). It is the same gate as $H$ in the quotation. Gate names in the code and in plot legends remain `"H"`.

<!--
```
q: |0⟩ ──[ * ]──[M]──   （* は H_Gate, Rx(π/2), Ry(π/2) のいずれか）
```
-->

```
q: |0⟩ ──[ * ]──[M]──   (* is one of H_Gate, Rx(π/2), Ry(π/2))
```

<!--
論文が挙げる C1 回路の特徴は次のとおりです。
- **長所**：どの量子コンピュータでも実行できる。1 量子ビットしか使わないので、最も性能の良い量子ビットを選べる。隣の量子ビットからの干渉（crosstalk）の影響が小さい。
- **短所**：1 回の測定（1 shot）で 1 ビットしか得られないので、生成速度が遅い。
-->

The paper lists the following characteristics of the C1 circuit.
- **Advantages**: It can be run on any quantum computer. Because it uses only one qubit, the best-performing qubit can be chosen. It is little affected by interference from neighboring qubits (crosstalk).
- **Disadvantage**: Each measurement (1 shot) yields only one bit, so the generation rate is low.

---

<!-- ### B. 量子乱数生成の原理 -->

### B. Principle of quantum random number generation

<!-- 1 量子ビットの状態は $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$（$|\alpha|^2 + |\beta|^2 = 1$）と書けます。これを計算基底（$Z$ 基底）で測定すると、**ボルンの規則**により -->

The state of one qubit can be written $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ ($|\alpha|^2 + |\beta|^2 = 1$). When it is measured in the computational basis ($Z$ basis), the **Born rule** gives 0 or 1 with probabilities

$$
P(0) = |\alpha|^2, \qquad P(1) = |\beta|^2
$$

<!-- の確率で 0 または 1 が得られます。測定結果は量子力学的に確率的で、原理的に予測できません。これが量子乱数の源です。 -->

The measurement outcome is probabilistic in the quantum-mechanical sense and is unpredictable in principle. This is the source of quantum randomness.

<!-- C1 回路で使う 3 つのゲートの行列表示は次のとおりです。 -->

The matrix representations of the three gates used in the C1 circuit are as follows.

$$
H_{\mathrm{Gate}} = \frac{1}{\sqrt{2}}\begin{pmatrix} 1 & 1 \\ 1 & -1 \end{pmatrix}, \quad
R_X(\theta) = \begin{pmatrix} \cos\frac{\theta}{2} & -i\sin\frac{\theta}{2} \\ -i\sin\frac{\theta}{2} & \cos\frac{\theta}{2} \end{pmatrix}, \quad
R_Y(\theta) = \begin{pmatrix} \cos\frac{\theta}{2} & -\sin\frac{\theta}{2} \\ \sin\frac{\theta}{2} & \cos\frac{\theta}{2} \end{pmatrix}
$$

<!-- これらを $|0\rangle$ に作用させると（論文の式と同じ）、 -->

Applying these to $|0\rangle$ gives (the same equations as in the paper)

$$
H_{\mathrm{Gate}}|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}, \qquad
R_X(\pi/2)|0\rangle = \frac{|0\rangle - i|1\rangle}{\sqrt{2}}, \qquad
R_Y(\pi/2)|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}
$$

<!-- となります。どれも $|\alpha|^2 = |\beta|^2 = 1/2$ なので、理想的には $P(0) = P(1) = 1/2$ です。ブロッホ球で言えば、3 つとも北極 $|0\rangle$ から赤道上の点へ状態を移すゲートです。 -->

In every case $|\alpha|^2 = |\beta|^2 = 1/2$, so ideally $P(0) = P(1) = 1/2$. On the Bloch sphere, all three gates move the state from the north pole $|0\rangle$ to a point on the equator.

<!-- 数学的には同じ確率分布を与えますが、論文は「**理論上は同じ回路でも、実機では振る舞いが異なりうる**」点を強調しています。たとえば $H_{\mathrm{Gate}}$ は IQM Spark のネイティブゲートではないため、複数のゲートに分解して実行されます（D を参照）。 -->

Although they give mathematically identical probability distributions, the paper emphasizes that "**circuits that are the same in theory can behave differently on real hardware**." For example, $H_{\mathrm{Gate}}$ is not a native gate of IQM Spark, so it is decomposed into multiple gates for execution (see D).

<!-- 各 shot は独立に測定されるので、理想的には「表と裏が等確率のコインを独立に投げた結果」と同じビット列（i.i.d. な公平ビット列）が得られます。 -->

Since each shot is measured independently, the ideal result is a bit string equivalent to "independent tosses of a fair coin" (an i.i.d. fair bit string).

---

<!-- ### C. 評価指標 -->

### C. Evaluation metrics

<!-- $N$ 回測定して 0 が $n_0$ 回、1 が $n_1$ 回出たとします。確率は相対頻度で推定します。 -->

Suppose that $N$ measurements yield 0 a total of $n_0$ times and 1 a total of $n_1$ times. The probabilities are estimated by relative frequencies.

$$
\hat{P}(0) = \frac{n_0}{N}, \qquad \hat{P}(1) = \frac{n_1}{N}
$$

<!-- **bias（偏り）**：理想値 $1/2$ からのずれです。 -->

**Bias**: the deviation from the ideal value $1/2$.

$$
\mathrm{bias} = \left| \hat{P}(1) - \frac{1}{2} \right|
$$

<!-- **Shannon エントロピー**：1 ビットあたりの平均的な情報量です。 -->

**Shannon entropy**: the average information content per bit.

$$
H_{\mathrm{Shannon}} = -\hat{P}(0)\log_2 \hat{P}(0) - \hat{P}(1)\log_2 \hat{P}(1)
$$

<!-- **min-entropy**：最も出やすい値を当てる確率 $\max(\hat{P}(0), \hat{P}(1))$ で決まる、最も保守的なエントロピーです。 -->

**Min-entropy**: the most conservative entropy, determined by the probability $\max(\hat{P}(0), \hat{P}(1))$ of guessing the most likely value.

$$
H_{\min} = -\log_2 \max\bigl(\hat{P}(0), \hat{P}(1)\bigr)
$$

<!-- 一般に $0 \le H_{\min} \le H_{\mathrm{Shannon}} \le 1$ で、公平なコインなら両方とも 1 になります。暗号では「攻撃者が出力を言い当てる確率」が問題になるので、乱数抽出（Toeplitz や Trevisan）の出力長は min-entropy から決めます。 -->

In general $0 \le H_{\min} \le H_{\mathrm{Shannon}} \le 1$, and both equal 1 for a fair coin. Because cryptography is concerned with "the probability that an attacker guesses the output," the output length of randomness extraction (Toeplitz or Trevisan) is determined from the min-entropy.

<!-- > **注意**：ここでの min-entropy は「各ビットが独立で同じ分布に従う（i.i.d.）」と仮定して頻度から計算したものです。相関などを考慮した厳密な評価（NIST SP 800-90B）は別のノートブックで扱います。 -->

> **Note**: The min-entropy here is computed from frequencies under the assumption that "each bit is independent and identically distributed (i.i.d.)." A rigorous assessment that accounts for correlations and similar effects (NIST SP 800-90B) is covered in a separate notebook.

<!-- **統計的な揺らぎ**：理想的な公平ビットでも、有限回の測定では $\hat{P}(1)$ は $1/2$ からずれます。$\hat{P}(1)$ の標準偏差は $1/(2\sqrt{N})$ なので、bias の期待値はおよそ -->

**Statistical fluctuation**: Even for ideal fair bits, $\hat{P}(1)$ deviates from $1/2$ for a finite number of measurements. The standard deviation of $\hat{P}(1)$ is $1/(2\sqrt{N})$, so the expected bias is approximately

$$
\mathbb{E}[\mathrm{bias}] \approx \sqrt{\frac{2}{\pi}} \cdot \frac{1}{2\sqrt{N}}
$$

<!-- です。 -->

<!-- $N = 10^6$ では約 $0.0004$ になります。また $H_{\min} = -\log_2(1/2 + \mathrm{bias}) \approx 1 - \frac{2}{\ln 2}\,\mathrm{bias}$ なので、$N = 1024$ では理想的な回路でも $H_{\min} \approx 0.96$ 程度になります。**測定回数が少ないと、回路が完璧でも min-entropy は 1 より小さく見える**ことに注意してください。 -->

For $N = 10^6$ this is about $0.0004$. Also, since $H_{\min} = -\log_2(1/2 + \mathrm{bias}) \approx 1 - \frac{2}{\ln 2}\,\mathrm{bias}$, even an ideal circuit gives only $H_{\min} \approx 0.96$ or so at $N = 1024$. Note that **with few measurements, the min-entropy appears smaller than 1 even if the circuit is perfect**.

---

<!-- ### D. ネイティブゲートと transpile -->

### D. Native gates and transpilation

<!-- 実際の量子コンピュータが直接実行できるゲートは限られており、これを**ネイティブゲート**と呼びます。論文の Table II によると、IQM Spark（Odra 5）のネイティブゲートは次のとおりです。 -->

The gates a real quantum computer can execute directly are limited; these are called **native gates**. According to Table II of the paper, the native gates of IQM Spark (Odra 5) are as follows.

<!--
| 種類 | ネイティブゲート |
|---|---|
| 1 量子ビット | $R_X$, $R_Y$ |
| 2 量子ビット | $\mathrm{CZ}$ |
-->

| Type | Native gates |
|---|---|
| 1-qubit | $R_X$, $R_Y$ |
| 2-qubit | $\mathrm{CZ}$ |

<!-- それ以外のゲートは、ネイティブゲートの組み合わせに変換してから実行します。この変換を **transpile** と呼びます。Qiskit の transpile では、$H_{\mathrm{Gate}}$ は次のように 2 個のゲートに分解されます。 -->

Any other gate is converted into a combination of native gates before execution. This conversion is called **transpilation**. In Qiskit's transpiler, $H_{\mathrm{Gate}}$ is decomposed into the following two gates.

$$
H_{\mathrm{Gate}} = e^{i\pi/2}\, R_X(\pi)\, R_Y(\pi/2)
$$

<!-- （右から順に作用するので、先に $R_Y(\pi/2)$、次に $R_X(\pi)$ です。$e^{i\pi/2}$ は大域位相で、測定結果には影響しません。）一方 $R_X(\pi/2)$ と $R_Y(\pi/2)$ はそのまま 1 個のゲートで実行できます。したがって、このノートブックのノイズモデルでは **$H_{\mathrm{Gate}}$ はゲート 2 個分のノイズを受ける**ことになります。ただし、ノイズを受ける回数が多いことと、結果の偏りが大きいことは同じではありません。偏りは、ノイズの種類とゲートの順序によって変わります（5.9 を参照）。なお、この分解は Qiskit の変換結果です。論文は Hadamard ゲートが $R_X$ と $R_Y$ の組み合わせなどネイティブゲートの列に変換されると述べていますが、具体的な分解は示していません。実機でも同じ 2 個になるとは限りません。 -->

(Gates act from right to left, so $R_Y(\pi/2)$ comes first, then $R_X(\pi)$. The factor $e^{i\pi/2}$ is a global phase and does not affect the measurement results.) On the other hand, $R_X(\pi/2)$ and $R_Y(\pi/2)$ can each be executed as a single gate. Therefore, in the noise model of this notebook, **$H_{\mathrm{Gate}}$ receives the noise of two gates**. However, receiving noise more often is not the same as having a larger bias in the results. The bias depends on the type of noise and on the order of the gates (see 5.9). Note that this decomposition is the result of Qiskit's conversion. The paper states that the Hadamard gate is converted into a sequence of native gates such as a combination of $R_X$ and $R_Y$, but it does not give a specific decomposition. Real hardware will not necessarily use the same two gates.

<!-- このノートブックでは、シミュレータでも基底ゲートを `["rx", "ry", "cz"]` に制限して transpile します。C1 回路は 2 量子ビットゲートを使わないので `cz` は使われませんが、C2・C3 回路のノートブックと設定をそろえるために入れておきます。 -->

In this notebook, even on the simulator, we transpile with the basis gates restricted to `["rx", "ry", "cz"]`. The C1 circuit uses no 2-qubit gates, so `cz` is not used, but it is included to keep the settings consistent with the C2 and C3 circuit notebooks.

---

<!-- ### E. ノイズモデル -->

### E. Noise model

<!-- 論文の Table II の値（Odra 5 の典型値）を使って、実機のノイズを模したモデルを作ります。論文から借りているのは数値だけで、ノイズの種類の組み合わせ（下の (a)〜(c)）と、読み出し誤差を対称とする仮定は、このノートブックで選んだものです。 -->

Using the values in Table II of the paper (typical values for Odra 5), we build a model that mimics the noise of the real hardware. Only the numerical values are borrowed from the paper; the combination of noise types ((a)–(c) below) and the assumption of symmetric readout error were chosen for this notebook.

<!--
| パラメータ | 論文の値 | このノートブックでの設定 |
|---|---|---|
| 1 量子ビットゲートの忠実度 | $\ge 99.9\%$ | 脱分極パラメータ $\lambda = 0.00198$（平均ゲート忠実度が 99.9% になる値） |
| 読み出しの忠実度 | $\ge 97\%$ | 読み出し誤差 $\varepsilon = 0.03$ |
| 1 量子ビットゲートの時間 | $\le 20$ ns | $t_g = 20$ ns |
| 緩和時間 $T_1$ | 0.964 ms | $T_1 = 0.964 \times 10^6$ ns |
| 位相緩和時間 $T_2$ | 1.155 ms | $T_2 = 1.155 \times 10^6$ ns |
-->

| Parameter | Value in the paper | Setting in this notebook |
|---|---|---|
| 1-qubit gate fidelity | $\ge 99.9\%$ | Depolarizing parameter $\lambda = 0.00198$ (the value that gives an average gate fidelity of 99.9%) |
| Readout fidelity | $\ge 97\%$ | Readout error $\varepsilon = 0.03$ |
| 1-qubit gate time | $\le 20$ ns | $t_g = 20$ ns |
| Relaxation time $T_1$ | 0.964 ms | $T_1 = 0.964 \times 10^6$ ns |
| Dephasing time $T_2$ | 1.155 ms | $T_2 = 1.155 \times 10^6$ ns |

<!-- 加えるノイズは次の 3 種類です。 -->

We add the following three kinds of noise.

<!-- **(a) 脱分極誤差（depolarizing error）**：確率 $\lambda$ で状態が完全にランダムな状態（最大混合状態 $I/2$）に置き換わるノイズです。 -->

**(a) Depolarizing error**: noise that replaces the state with a completely random state (the maximally mixed state $I/2$) with probability $\lambda$.

$$
\mathcal{E}_{\mathrm{dep}}(\rho) = (1-\lambda)\,\rho + \lambda\,\frac{I}{2}
$$

<!-- **(b) 熱緩和誤差（thermal relaxation error）**：ゲートを実行している時間 $t_g$ の間に起こる、エネルギー緩和（$|1\rangle \to |0\rangle$、時定数 $T_1$）と位相緩和（時定数 $T_2$）です。$|1\rangle$ が $|0\rangle$ に落ちる確率は $\gamma = 1 - e^{-t_g/T_1} \approx 2 \times 10^{-5}$ です。このモデルの熱緩和はゲート中だけで、読み出し中の緩和は含みません。 -->

**(b) Thermal relaxation error**: energy relaxation ($|1\rangle \to |0\rangle$, time constant $T_1$) and dephasing (time constant $T_2$) occurring during the gate time $t_g$. The probability that $|1\rangle$ decays to $|0\rangle$ is $\gamma = 1 - e^{-t_g/T_1} \approx 2 \times 10^{-5}$. In this model, thermal relaxation occurs only during gates; relaxation during readout is not included.

<!-- (a) と (b) を合成（compose）したものを、すべての $R_X$, $R_Y$ ゲートの後に加えます。これを「ゲートノイズ」と呼びます。 -->

The composition of (a) and (b) is applied after every $R_X$ and $R_Y$ gate. We call this the "gate noise."

<!-- **$\lambda$ の決め方**：$\lambda$ は平均ゲート不忠実度そのものではありません。1 量子ビットの脱分極誤差の平均ゲート不忠実度は $\lambda/2$ です。これに (b) の熱緩和の分（約 $9.2 \times 10^{-6}$）を加えた合計が論文の値 $0.001$（忠実度 99.9%）になるように、$\lambda = 2\,(0.001 - 9.2 \times 10^{-6}) \approx 0.00198$ とします。 -->

**How $\lambda$ is chosen**: $\lambda$ is not the average gate infidelity itself. The average gate infidelity of a 1-qubit depolarizing error is $\lambda/2$. We set $\lambda = 2\,(0.001 - 9.2 \times 10^{-6}) \approx 0.00198$ so that the total, including the contribution of the thermal relaxation (b) (about $9.2 \times 10^{-6}$), equals the paper's value of $0.001$ (fidelity 99.9%).

<!-- **(c) 読み出し誤差（readout error）**：本当は 0 なのに 1 と読む（またはその逆）誤りです。行を「本当の状態」、列を「読み出した値」とした混同行列で表します。 -->

**(c) Readout error**: an error in which a true 0 is read as 1 (or vice versa). It is represented by a confusion matrix whose rows are the "true state" and whose columns are the "value read out."

$$
\begin{pmatrix} P(0|0) & P(1|0) \\ P(0|1) & P(1|1) \end{pmatrix}
= \begin{pmatrix} 1-\varepsilon & \varepsilon \\ \varepsilon & 1-\varepsilon \end{pmatrix}
$$

<!--
**ノイズが結果に与える影響の予想**：
- 脱分極誤差は状態を $I/2$ に近づけるだけなので、$P(1) = 1/2$ は変わりません。
- 読み出し誤差が対称（$P(1|0) = P(0|1) = \varepsilon$）なら、$P(1)' = (1-\varepsilon)\cdot\frac{1}{2} + \varepsilon\cdot\frac{1}{2} = \frac{1}{2}$ で、やはり偏りは生じません。
- 熱緩和だけによる偏りは、$R_X(\pi/2)$, $R_Y(\pi/2)$ では $\gamma/2 \approx 1 \times 10^{-5}$、$H_{\mathrm{Gate}}$（$R_Y(\pi/2) \to R_X(\pi)$）では $\gamma^2/2 \approx 2 \times 10^{-10}$ です（違いの理由は 5.9 を参照）。どちらも $N = 10^6$ のときの統計的な揺らぎ（約 $4 \times 10^{-4}$）よりずっと小さいです。
-->

**Expected effects of the noise on the results**:
- Depolarizing error only pushes the state toward $I/2$, so $P(1) = 1/2$ does not change.
- If the readout error is symmetric ($P(1|0) = P(0|1) = \varepsilon$), then $P(1)' = (1-\varepsilon)\cdot\frac{1}{2} + \varepsilon\cdot\frac{1}{2} = \frac{1}{2}$, so again no bias arises.
- The bias from thermal relaxation alone is $\gamma/2 \approx 1 \times 10^{-5}$ for $R_X(\pi/2)$ and $R_Y(\pi/2)$, and $\gamma^2/2 \approx 2 \times 10^{-10}$ for $H_{\mathrm{Gate}}$ ($R_Y(\pi/2) \to R_X(\pi)$) (see 5.9 for the reason for the difference). Both are far smaller than the statistical fluctuation at $N = 10^6$ (about $4 \times 10^{-4}$).

<!-- したがって、このノイズモデルでは **bias は統計的な揺らぎの範囲に収まる**と予想されます。もし読み出し誤差が非対称（実機では $P(0|1) > P(1|0)$ が多い）なら、$P(1)' = \frac{1}{2}\bigl(1 - P(0|1) + P(1|0)\bigr)$ となり偏りが生じます。 -->

Therefore, in this noise model, **the bias is expected to stay within the range of statistical fluctuation**. If the readout error were asymmetric (on real hardware, often $P(0|1) > P(1|0)$), then $P(1)' = \frac{1}{2}\bigl(1 - P(0|1) + P(1|0)\bigr)$ and a bias would arise.

<!-- この予想は、対称な読み出し誤差を仮定した**このモデルに限った話**です。論文は Odra 5 の実機で 1 の割合が全体に 50% より低くなる傾向を報告し、$P(0|1) > P(1|0)$ で説明できるとしています。校正データの平均から見積もった C1 の 1 の割合は約 49.09%（bias 約 0.009）で、$N = 10^6$ の揺らぎよりずっと大きい値です。このノートブックのノイズモデルでは、この偏りは再現されません。 -->

This expectation applies **only to this model**, which assumes symmetric readout error. The paper reports a tendency on the real Odra 5 hardware for the fraction of 1s to fall below 50% overall, and says this can be explained by $P(0|1) > P(1|0)$. The fraction of 1s for C1 estimated from the average of the calibration data is about 49.09% (bias about 0.009), far larger than the fluctuation at $N = 10^6$. The noise model in this notebook does not reproduce this bias.

<!-- このノートブックでは、ノイズの種類を次の 3 つから選べます。 -->

In this notebook, the type of noise can be chosen from the following three options.

<!--
| `NOISE_KIND` | 加えるノイズ |
|---|---|
| `"gate"` | (a) + (b) のゲートノイズだけ |
| `"readout"` | (c) の読み出し誤差だけ |
| `"both"` | ゲートノイズと読み出し誤差の両方 |
-->

| `NOISE_KIND` | Noise added |
|---|---|
| `"gate"` | Only the gate noise (a) + (b) |
| `"readout"` | Only the readout error (c) |
| `"both"` | Both the gate noise and the readout error |

---

<!-- ### F. 使用する Qiskit の関数・クラス -->

### F. Qiskit functions and classes used

<!--
| 関数・クラス | パッケージ | 役割 |
|---|---|---|
| `QuantumCircuit(1, 1)` | `qiskit` | 量子ビット 1 個・古典ビット 1 個の回路を作る |
| `qc.h(0)`, `qc.rx(θ, 0)`, `qc.ry(θ, 0)` | `qiskit` | 量子ビット 0 にゲートを作用させる |
| `qc.measure(0, 0)` | `qiskit` | 量子ビット 0 を測定し、古典ビット 0 に記録する |
| `qc.draw()` | `qiskit` | 回路図を表示する |
| `transpile(qc, basis_gates=...)` | `qiskit` | 回路を指定したゲートだけを使う回路に変換する |
| `Operator(qc).equiv(...)` | `qiskit.quantum_info` | 2 つの回路が（大域位相を除いて）同じ演算か確かめる |
| `AerSimulator(noise_model=...)` | `qiskit_aer` | 量子回路のシミュレータ。ノイズモデルを渡せる |
| `backend.run(qc, shots=N, memory=True)` | `qiskit_aer` | 回路を $N$ 回測定する。`memory=True` で各 shot の結果を順番に保存する |
| `result.get_counts()` | `qiskit` | 0 と 1 が出た回数の辞書（例 `{'0': 507, '1': 517}`）を返す |
| `result.get_memory()` | `qiskit` | 各 shot の測定結果を測定順のリストで返す（生のビット列） |
| `NoiseModel(basis_gates=...)` | `qiskit_aer.noise` | ノイズモデルの入れ物を作る |
| `depolarizing_error(λ, 1)` | `qiskit_aer.noise` | 1 量子ビットの脱分極誤差 |
| `thermal_relaxation_error(T1, T2, t)` | `qiskit_aer.noise` | 時間 $t$ の熱緩和誤差（$T_1$, $T_2$, $t$ は同じ単位） |
| `err1.compose(err2)` | `qiskit_aer.noise` | 2 つの誤差を順に合成する |
| `ReadoutError([[...], [...]])` | `qiskit_aer.noise` | 混同行列で読み出し誤差を作る |
| `noise_model.add_all_qubit_quantum_error(err, ["rx", "ry"])` | `qiskit_aer.noise` | 指定したゲートの後に、すべての量子ビットで誤差を加える |
| `noise_model.add_all_qubit_readout_error(err)` | `qiskit_aer.noise` | すべての量子ビットの測定に読み出し誤差を加える |
-->

| Function / class | Package | Role |
|---|---|---|
| `QuantumCircuit(1, 1)` | `qiskit` | Creates a circuit with 1 qubit and 1 classical bit |
| `qc.h(0)`, `qc.rx(θ, 0)`, `qc.ry(θ, 0)` | `qiskit` | Applies a gate to qubit 0 |
| `qc.measure(0, 0)` | `qiskit` | Measures qubit 0 and records the result in classical bit 0 |
| `qc.draw()` | `qiskit` | Displays the circuit diagram |
| `transpile(qc, basis_gates=...)` | `qiskit` | Converts a circuit into one that uses only the specified gates |
| `Operator(qc).equiv(...)` | `qiskit.quantum_info` | Checks whether two circuits implement the same operation (up to a global phase) |
| `AerSimulator(noise_model=...)` | `qiskit_aer` | Quantum circuit simulator. A noise model can be passed in |
| `backend.run(qc, shots=N, memory=True)` | `qiskit_aer` | Measures the circuit $N$ times. `memory=True` stores the result of each shot in order |
| `result.get_counts()` | `qiskit` | Returns a dictionary of how many times 0 and 1 occurred (e.g., `{'0': 507, '1': 517}`) |
| `result.get_memory()` | `qiskit` | Returns the result of each shot as a list in measurement order (the raw bit string) |
| `NoiseModel(basis_gates=...)` | `qiskit_aer.noise` | Creates a container for a noise model |
| `depolarizing_error(λ, 1)` | `qiskit_aer.noise` | 1-qubit depolarizing error |
| `thermal_relaxation_error(T1, T2, t)` | `qiskit_aer.noise` | Thermal relaxation error for time $t$ ($T_1$, $T_2$, and $t$ in the same unit) |
| `err1.compose(err2)` | `qiskit_aer.noise` | Composes two errors in sequence |
| `ReadoutError([[...], [...]])` | `qiskit_aer.noise` | Creates a readout error from a confusion matrix |
| `noise_model.add_all_qubit_quantum_error(err, ["rx", "ry"])` | `qiskit_aer.noise` | Adds an error after the specified gates on all qubits |
| `noise_model.add_all_qubit_readout_error(err)` | `qiskit_aer.noise` | Adds a readout error to the measurement of all qubits |

---

<!-- ### G. このノートブックの使い方 -->

### G. How to use this notebook

<!-- 上のセルから順に実行してください。シミュレーションの条件は、次の変数を書き換えて選びます。 -->

Run the cells in order from the top. The simulation conditions are selected by editing the following variables.

<!--
| 変数 | 意味 | 選択肢・既定値 |
|---|---|---|
| `SAVE_RESULTS` | 4.3〜5.8 の raw data と結果を保存するか（1. のセル） | `True`（`results/c1/<日時>/` に保存）または `False`（保存しない。既定） |
| `SAVE_SWEEP` | 5.9 のノイズの強さを変える実験の raw data と結果を保存するか（1. のセル） | `True`（同じ日時のフォルダに保存）または `False`（保存しない。既定） |
| `GATE_IDEAL` | ノイズなしで使うゲート | `"H"`, `"Rx"`, `"Ry"`（既定 `"H"`） |
| `SHOTS_IDEAL` | ノイズなしの測定回数 | 既定 `1_000_000` |
| `GATE_NOISY` | ノイズありで使うゲート | `"H"`, `"Rx"`, `"Ry"`（既定 `"H"`） |
| `NOISE_KIND` | 加えるノイズの種類 | `"gate"`, `"readout"`, `"both"`（既定 `"both"`） |
| `SHOTS_NOISY` | ノイズありの測定回数 | 既定 `1_000_000` |
| `N_TRIALS` | 繰り返しの回数 | 既定 `10` |
| `SEED` | シミュレータの乱数シード | `None`（毎回違う結果）または整数（結果を再現できる） |
-->

| Variable | Meaning | Options / default |
|---|---|---|
| `SAVE_RESULTS` | Whether to save the raw data and results of 4.3–5.8 (cell in 1.) | `True` (save in `results/c1/<date_time>/`) or `False` (do not save; default) |
| `SAVE_SWEEP` | Whether to save the raw data and results of the noise-strength sweep in 5.9 (cell in 1.) | `True` (save in the same date-and-time folder) or `False` (do not save; default) |
| `GATE_IDEAL` | Gate used in the noiseless case | `"H"`, `"Rx"`, `"Ry"` (default `"H"`) |
| `SHOTS_IDEAL` | Number of noiseless measurements | Default `1_000_000` |
| `GATE_NOISY` | Gate used in the noisy case | `"H"`, `"Rx"`, `"Ry"` (default `"H"`) |
| `NOISE_KIND` | Type of noise added | `"gate"`, `"readout"`, `"both"` (default `"both"`) |
| `SHOTS_NOISY` | Number of noisy measurements | Default `1_000_000` |
| `N_TRIALS` | Number of repetitions | Default `10` |
| `SEED` | Random seed of the simulator | `None` (different results each time) or an integer (results are reproducible) |

<!-- #### 保存されるファイル -->

#### Saved files

<!-- 1. のセルで `SAVE_RESULTS = True` または `SAVE_SWEEP = True` にすると、そのセルを実行した日時の名前のフォルダ `results/c1/<年-月-日_時分秒>/`（変数 `RUN_DIR`）を作ります。`SAVE_RESULTS = True` なら 4.3〜5.8 の raw data と結果を、`SAVE_SWEEP = True` なら 5.9 の raw data と結果を、その中に保存します。5.9 は保存量が多いので、別のスイッチにしています。どちらも `False`（既定）なら何も保存しません。1. のセルを実行し直すたびに新しいフォルダになるので、前の結果を上書きすることはありません。 -->

If you set `SAVE_RESULTS = True` or `SAVE_SWEEP = True` in the cell in 1., a folder named after the date and time that cell was run, `results/c1/<year-month-day_hourminutesecond>/` (variable `RUN_DIR`), is created. `SAVE_RESULTS = True` saves the raw data and results of 4.3–5.8 in it, and `SAVE_SWEEP = True` saves those of 5.9. Because 5.9 produces a lot of data, it has its own switch. If both are `False` (the default), nothing is saved. Each time you rerun the cell in 1., a new folder is used, so earlier results are never overwritten.

<!-- `RUN_DIR` の中は、実験（節）ごとのフォルダに分かれています。フォルダ名の先頭の数字は、ノートブックの節の番号です。 -->

Inside `RUN_DIR`, there is one folder per experiment (section). The number at the start of each folder name is the section number in this notebook.

<!--
| フォルダ | 節 | 中身 |
|---|---|---|
| `4.3_ideal_single/` | 4.3 | ノイズなしの 1 回の測定の raw data |
| `4.5_ideal_trials/` | 4.5 | ノイズなしの繰り返し測定の raw data（試行ごとに 1 ファイル）と、全試行の一覧 `trials.csv` |
| `4.6_ideal_gates/` | 4.6 | 3 つのゲートの raw data と、その一覧 `summary.csv` |
| `5.5_noisy_single/` | 5.5 | ノイズありの 1 回の測定の raw data |
| `5.7_noisy_trials/` | 5.7 | ノイズありの繰り返し測定の raw data（試行ごとに 1 ファイル）と `trials.csv` |
| `5.8_noisy_comparison/` | 5.8 | ゲート × ノイズの種類の 9 通りの raw data と `summary.csv` |
| `5.9_noise_sweep/` | 5.9 | ノイズの強さを変えた各条件・各試行の raw data（ノイズの種類ごとのフォルダ）と、`trials.csv`・`summary.csv`（`SAVE_SWEEP = True` のときだけ） |
-->

| Folder | Section | Contents |
|---|---|---|
| `4.3_ideal_single/` | 4.3 | Raw data of the single noiseless measurement |
| `4.5_ideal_trials/` | 4.5 | Raw data of the repeated noiseless measurements (one file per trial) and the list of all trials, `trials.csv` |
| `4.6_ideal_gates/` | 4.6 | Raw data for the three gates and their list, `summary.csv` |
| `5.5_noisy_single/` | 5.5 | Raw data of the single noisy measurement |
| `5.7_noisy_trials/` | 5.7 | Raw data of the repeated noisy measurements (one file per trial) and `trials.csv` |
| `5.8_noisy_comparison/` | 5.8 | Raw data for the 9 combinations of gate × noise type and `summary.csv` |
| `5.9_noise_sweep/` | 5.9 | Raw data for each condition and trial with varying noise strength (one folder per noise type), `trials.csv`, and `summary.csv` (only when `SAVE_SWEEP = True`) |

<!-- 既定の設定で両方を保存すると、次のようになります。 -->

With the default settings and both switches on, the saved files look like this.

<!--
```
results/c1/
└── 2026-11-10_153012/                 RUN_DIR（1. のセルを実行した日時）
    ├── 4.3_ideal_single/
    │   ├── c1_H_ideal.txt
    │   └── c1_H_ideal.txt.json
    ├── 4.5_ideal_trials/
    │   ├── c1_H_ideal_trial001.txt
    │   ├── c1_H_ideal_trial001.txt.json
    │   ├── ...
    │   └── trials.csv
    ├── 4.6_ideal_gates/
    │   ├── c1_H_ideal.txt
    │   ├── c1_H_ideal.txt.json
    │   ├── c1_Rx_ideal.txt ...
    │   └── summary.csv
    ├── 5.5_noisy_single/
    │   ├── c1_H_noise-both.txt
    │   └── c1_H_noise-both.txt.json
    ├── 5.7_noisy_trials/
    │   ├── c1_H_noise-both_trial001.txt
    │   ├── c1_H_noise-both_trial001.txt.json
    │   ├── ...
    │   └── trials.csv
    ├── 5.8_noisy_comparison/
    │   ├── c1_H_noise-gate.txt
    │   ├── c1_H_noise-gate.txt.json
    │   ├── c1_H_noise-readout.txt ...
    │   └── summary.csv
    └── 5.9_noise_sweep/
        ├── trials.csv
        ├── summary.csv
        ├── depolarizing/
        │   ├── c1_H_depolarizing-0_trial001.txt
        │   ├── c1_H_depolarizing-0_trial001.txt.json
        │   └── c1_H_depolarizing-0.01_trial001.txt ...
        └── ...
```
-->

```
results/c1/
└── 2026-11-10_153012/                 RUN_DIR (date and time the cell in 1. was run)
    ├── 4.3_ideal_single/
    │   ├── c1_H_ideal.txt
    │   └── c1_H_ideal.txt.json
    ├── 4.5_ideal_trials/
    │   ├── c1_H_ideal_trial001.txt
    │   ├── c1_H_ideal_trial001.txt.json
    │   ├── ...
    │   └── trials.csv
    ├── 4.6_ideal_gates/
    │   ├── c1_H_ideal.txt
    │   ├── c1_H_ideal.txt.json
    │   ├── c1_Rx_ideal.txt ...
    │   └── summary.csv
    ├── 5.5_noisy_single/
    │   ├── c1_H_noise-both.txt
    │   └── c1_H_noise-both.txt.json
    ├── 5.7_noisy_trials/
    │   ├── c1_H_noise-both_trial001.txt
    │   ├── c1_H_noise-both_trial001.txt.json
    │   ├── ...
    │   └── trials.csv
    ├── 5.8_noisy_comparison/
    │   ├── c1_H_noise-gate.txt
    │   ├── c1_H_noise-gate.txt.json
    │   ├── c1_H_noise-readout.txt ...
    │   └── summary.csv
    └── 5.9_noise_sweep/
        ├── trials.csv
        ├── summary.csv
        ├── depolarizing/
        │   ├── c1_H_depolarizing-0_trial001.txt
        │   ├── c1_H_depolarizing-0_trial001.txt.json
        │   └── c1_H_depolarizing-0.01_trial001.txt ...
        └── ...
```

<!--
raw data のファイル名は `<回路>_<ゲート>_<条件>[_trial<試行番号>].txt` です。

- 回路：`c1`
- ゲート：`H`, `Rx`, `Ry`
- 条件：`ideal`（ノイズなし）、`noise-gate`・`noise-readout`・`noise-both`（5.3 の `NOISE_KIND`）、5.9 では `<ノイズの種類>-<強さ>`（例：`readout-0.04`、`thermal-2500`）。5.9 のノイズの種類は `depolarizing`, `thermal`, `readout` です
- 試行番号：繰り返し測定（4.5・5.7・5.9）だけにつく、1 から数えた 3 桁の番号
-->

Raw-data files are named `<circuit>_<gate>_<condition>[_trial<trial number>].txt`.

- Circuit: `c1`
- Gate: `H`, `Rx`, `Ry`
- Condition: `ideal` (no noise); `noise-gate`, `noise-readout`, `noise-both` (`NOISE_KIND` in 5.3); in 5.9, `<noise type>-<strength>` (e.g., `readout-0.04`, `thermal-2500`). The noise types in 5.9 are `depolarizing`, `thermal`, `readout`
- Trial number: only for repeated measurements (4.5, 5.7, 5.9); a 3-digit number counted from 1

<!--
| ファイル | 中身 |
|---|---|
| `<名前>.txt` | `'0'` と `'1'` の文字を改行なしで並べたテキスト（1 文字 = 1 shot）。測定順のビット列。SP800_90B・Toeplitz・Trevisan・SP800_22 のノートブックの `load_bits` で読める |
| `<名前>.txt.json` | 測定の条件と結果。キーは `circuit`, `gate`, `shots`, `seed`, `metadata`（条件・試行番号など）, `basis_gates`, `optimization_level`, `method`, `qiskit_version`, `qiskit_aer_version`, `noise_model`（`NoiseModel.to_dict` の内容。ノイズなしなら `null`）, `counts`, `evaluation`（`evaluate` の戻り値） |
| `trials.csv`（4.5・5.7） | 1 試行 1 行。列は `file`（raw data のファイル名）, `gate`, `condition`, `trial`, `seed` と、`evaluate` の戻り値のうち数値の項目 |
| `summary.csv`（4.6・5.8） | 1 ファイル 1 行。列は `file`, `gate`, `condition`, `seed` と、`evaluate` の戻り値のうち数値の項目 |
| `5.9_noise_sweep/trials.csv` | 1 条件・1 試行 1 行。列は `component`, `strength`, `gate`, `trial`, `seed`, `shots`, `n0`, `n1`, `P0`, `P1`, `bias`, `shannon`, `min_entropy` |
| `5.9_noise_sweep/summary.csv` | 1 条件 1 行。列は `component`, `strength`, `gate`、密度行列から求めた `exact_P1`, `exact_bias`, `exact_min_entropy`、`P1`, `bias`, `shannon`, `min_entropy` の試行間の平均 `<指標>_mean` と標準偏差 `<指標>_std` |
-->

| File | Contents |
|---|---|
| `<name>.txt` | Text of the characters `'0'` and `'1'` with no line breaks (1 character = 1 shot): the bit string in measurement order. Can be read with `load_bits` in the SP800_90B, Toeplitz, Trevisan, and SP800_22 notebooks |
| `<name>.txt.json` | Measurement conditions and results. Keys: `circuit`, `gate`, `shots`, `seed`, `metadata` (condition, trial number, and so on), `basis_gates`, `optimization_level`, `method`, `qiskit_version`, `qiskit_aer_version`, `noise_model` (the contents of `NoiseModel.to_dict`; `null` without noise), `counts`, `evaluation` (the return value of `evaluate`) |
| `trials.csv` (4.5, 5.7) | One row per trial. Columns: `file` (the raw-data file name), `gate`, `condition`, `trial`, `seed`, and the numeric items of the return value of `evaluate` |
| `summary.csv` (4.6, 5.8) | One row per file. Columns: `file`, `gate`, `condition`, `seed`, and the numeric items of the return value of `evaluate` |
| `5.9_noise_sweep/trials.csv` | One row per condition and trial. Columns: `component`, `strength`, `gate`, `trial`, `seed`, `shots`, `n0`, `n1`, `P0`, `P1`, `bias`, `shannon`, `min_entropy` |
| `5.9_noise_sweep/summary.csv` | One row per condition. Columns: `component`, `strength`, `gate`; `exact_P1`, `exact_bias`, `exact_min_entropy` computed from the density matrix; and the mean `<metric>_mean` and standard deviation `<metric>_std` over trials of `P1`, `bias`, `shannon`, `min_entropy` |

<!--
- ファイルは上書きしません。同じ `RUN_DIR` のまま同じセルをもう一度実行すると、測定を始める前にエラー（`FileExistsError`）になります。もう一度保存するときは、1. のセルを実行し直して新しいフォルダを作ってください。
- CSV は、その実験のすべての試行・条件が終わったあとに、raw data と同じフォルダに書きます。CSV がないフォルダは、途中で止まった実験の raw data です。
- `run_c1(..., raw_path=...)` や `run_trials(..., raw_dir=...)` に自分で保存先を指定した場合は、`SAVE_RESULTS` に関係なくその場所に保存します（4.5 を参照）。
- 保存量の目安：1 ファイル約 1 MB（$10^6$ shots）です。既定の設定では、4.3〜5.8 の合計が約 34 MB、5.9（`SAVE_SWEEP = True` のとき）が約 280 MB です。
-->

- Files are never overwritten. If you run the same cell again with the same `RUN_DIR`, an error (`FileExistsError`) is raised before any measurement. To save again, rerun the cell in 1. to create a new folder.
- The CSV files are written in the same folder as the raw data, after all trials and conditions of that experiment have finished. A folder without CSV files holds the raw data of an experiment that stopped partway.
- If you give your own destination to `run_c1(..., raw_path=...)` or `run_trials(..., raw_dir=...)`, the files are saved there regardless of `SAVE_RESULTS` (see 4.5).
- Approximate size: Each file is about 1 MB ($10^6$ shots). With the default settings, 4.3–5.8 save about 34 MB in total, and 5.9 (when `SAVE_SWEEP = True`) saves about 280 MB.

<!-- ## 1. 準備 -->

## 1. Setup

<!-- 使うライブラリを読み込み、ネイティブゲート（説明パート D）を `BASIS_GATES` として定義します。 -->

We load the libraries used and define the native gates (Explanation Part D) as `BASIS_GATES`.

<!--
- `qiskit`：回路を作る・transpile する
- `qiskit_aer`：シミュレータとノイズモデル
- `IPython.display`：結果の表を Markdown で表示する
- `matplotlib`：5.9 のグラフを描く
-->

- `qiskit`: building and transpiling circuits
- `qiskit_aer`: simulator and noise models
- `IPython.display`: displaying result tables as Markdown
- `matplotlib`: drawing the plots in 5.9

<!-- **結果の保存**：このセルの 2 つのスイッチで、raw data と結果を保存するかどうかを選びます。`SAVE_RESULTS = True` で 4.3〜5.8 を、`SAVE_SWEEP = True` で 5.9（ノイズの強さを段階的に変える実験。保存量が多い）を保存します。どちらかを `True` にすると、このセルを実行した日時の名前のフォルダ（例：`results/c1/2026-11-10_153012/`）を作り、その中の実験ごとのフォルダに保存します。どちらも `False`（既定）なら何も保存しません。保存先と中身は「G. このノートブックの使い方」の「保存されるファイル」を参照してください。 -->

**Saving the results**: the two switches in this cell select whether the raw data and results are saved. `SAVE_RESULTS = True` saves 4.3–5.8, and `SAVE_SWEEP = True` saves 5.9 (the experiment that changes the noise strength step by step, which produces a lot of data). If either is `True`, a folder named after the date and time this cell was run (e.g., `results/c1/2026-11-10_153012/`) is created, and the files are saved in per-experiment folders inside it. If both are `False` (the default), nothing is saved. For the destinations and contents, see "Saved files" in "G. How to use this notebook".

<!-- セルは「回路の定数」「シミュレーションのシード」「保存のスイッチ」の 3 つのまとまりに分けています。保存に使う関数は、次の「保存に使う関数」のセルにまとめています。 -->

The cell is divided into three groups: circuit constants, the simulation seed, and the saving switches. The functions used for saving are collected in the following "Functions used for saving" cell.

In [ ]:
import csv
import json
import math
import statistics
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import qiskit
import qiskit_aer
from IPython.display import Markdown, display
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error, thermal_relaxation_error

print("qiskit     :", qiskit.__version__)
print("qiskit-aer :", qiskit_aer.__version__)

# ---- Circuit ----
BASIS_GATES = ["rx", "ry", "cz"]    # Native gates of IQM Spark (Odra 5) (arXiv:2512.09862, Table II)
GATES = ["H", "Rx", "Ry"]           # Gates that create the superposition

# ---- Simulation ----
SEED = None    # Random seed of the simulator (None: different results each time; integer: reproducible)

# ---- Saving (see "Saved files" in G) ----
SAVE_RESULTS = False    # Save the raw data and results of 4.3-5.8
SAVE_SWEEP = False      # Save the raw data and results of the noise-strength sweep in 5.9 (large)
RESULTS_ROOT = Path("results/c1")
RUN_DIR = RESULTS_ROOT / datetime.now().strftime("%Y-%m-%d_%H%M%S") if (SAVE_RESULTS or SAVE_SWEEP) else None

if RUN_DIR is None:
    print("save to    : nothing is saved")
else:
    print(f"save to    : {RUN_DIR} (4.3-5.8: {SAVE_RESULTS}, 5.9: {SAVE_SWEEP})")

<!-- **保存に使う関数** -->

**Functions used for saving**

<!-- raw data と CSV の保存は、次のセルの関数にまとめています。`run_c1`（4.5）や 4.3・5.5 のセルはこれらを呼び出すだけです。どのファイルも上書きしません。 -->

Saving the raw data and the CSV files is collected in the functions of the next cell. `run_c1` (4.5) and the cells in 4.3 and 5.5 only call them. No file is ever overwritten.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `output_dir` | 4.3〜5.8 の実験のフォルダ `RUN_DIR/<name>`（`SAVE_RESULTS = False` なら `None`） | `output_dir("4.5_ideal_trials")` |
| `raw_files` | 1 組の raw data の 2 つのファイルのパス（`.txt`, `.txt.json`） | `raw_files("results/c1/manual/c1_H_ideal.txt")` |
| `ensure_new` | どちらかのファイルがすでにあればエラー（測定の前に確かめるため） | `ensure_new("results/c1/manual/c1_H_ideal.txt")` |
| `save_raw` | 各 shot の測定結果（1 shot = 1 文字）のビット列と条件の JSON を保存し、`.txt` のパスを返す | `save_raw(path, result.get_memory(), "H", 1000, 2026)` |
| `write_csv` | 1 レコード 1 行の CSV を書き、パスを返す（数値と文字列の項目だけを列にする） | `write_csv(path, records)` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `output_dir` | Folder of an experiment in 4.3–5.8, `RUN_DIR/<name>` (`None` when `SAVE_RESULTS = False`) | `output_dir("4.5_ideal_trials")` |
| `raw_files` | Paths of the two files of one raw-data set (`.txt`, `.txt.json`) | `raw_files("results/c1/manual/c1_H_ideal.txt")` |
| `ensure_new` | Raises an error if either file already exists (checked before measuring) | `ensure_new("results/c1/manual/c1_H_ideal.txt")` |
| `save_raw` | Saves the bit string of the measurement results (1 shot = 1 character) and the JSON of conditions, and returns the path of the `.txt` | `save_raw(path, result.get_memory(), "H", 1000, 2026)` |
| `write_csv` | Writes a CSV with one row per record and returns its path (only numeric and string items become columns) | `write_csv(path, records)` |

<!-- `save_raw` の引数 `gate`, `shots`, `seed`, `noise_model`, `method`, `optimization_level`, `metadata` は、JSON に記録する測定の条件です。 -->

The arguments `gate`, `shots`, `seed`, `noise_model`, `method`, `optimization_level`, and `metadata` of `save_raw` are the measurement conditions recorded in the JSON.

In [ ]:
def output_dir(name):
    # Folder of one experiment in 4.3-5.8 (None when SAVE_RESULTS = False)
    return RUN_DIR / name if SAVE_RESULTS else None


def raw_files(path):
    # The two files of one raw-data set: the bit string and the conditions
    path = Path(path)
    return path, path.with_name(path.name + ".json")


def ensure_new(path):
    # Files are never overwritten: stop if either file already exists
    for p in raw_files(path):
        if p.exists():
            raise FileExistsError(f"Destination already exists: {p}. Run the Setup cell (1.) again to save in a new folder.")


def save_raw(path, memory, gate, shots, seed, noise_model=None, *,
             method="automatic", optimization_level=None, metadata=None):
    # memory: the result of each shot in measurement order (result.get_memory()); 1 shot = 1 character
    ensure_new(path)
    bits_path, info_path = raw_files(path)
    counts = dict(Counter(memory))
    info = {
        "circuit": "C1", "gate": gate, "shots": shots, "seed": seed, "metadata": metadata,
        "basis_gates": BASIS_GATES, "optimization_level": optimization_level, "method": method,
        "qiskit_version": qiskit.__version__, "qiskit_aer_version": qiskit_aer.__version__,
        "noise_model": None if noise_model is None else noise_model.to_dict(serializable=True),
        "counts": counts, "evaluation": evaluate(counts),
    }
    bits_path.parent.mkdir(parents=True, exist_ok=True)
    bits_path.write_text("".join(memory), encoding="ascii")
    info_path.write_text(json.dumps(info, indent=2) + "\n", encoding="utf-8")
    return bits_path


def write_csv(path, records):
    # One row per record; only scalar values become columns. Existing files are never overwritten.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    columns = [k for k, v in records[0].items() if isinstance(v, (str, int, float, bool, type(None)))]
    with path.open("x", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(records)
    return path

<!-- ## 2. 評価指標を計算する関数 -->

## 2. Functions for computing the evaluation metrics

<!-- 説明パート C の式を関数にします。 -->

We turn the formulas of Explanation Part C into functions.

<!--
- `shannon_entropy(p0, p1)`：$H_{\mathrm{Shannon}} = -p_0\log_2 p_0 - p_1\log_2 p_1$。$p = 0$ の項は $0 \log_2 0 = 0$ として扱います。
- `min_entropy(p0, p1)`：$H_{\min} = -\log_2 \max(p_0, p_1)$
- `evaluate(counts)`：測定回数の辞書 `counts` から $n_0, n_1, \hat{P}(0), \hat{P}(1)$, bias, $H_{\mathrm{Shannon}}$, $H_{\min}$ をまとめて計算する
- `expected_bias(shots)`：理想的な公平ビットで期待される bias $\sqrt{2/\pi}\,/\,(2\sqrt{N})$。結果と比べる目安に使います
- `show_metrics(r)`：P(0)・P(1)・bias・エントロピーと、公平なビットの目安を表示する（4.4・5.6 で使う）
- `show_table(rows, first_col)`：結果を表にして表示する
-->

- `shannon_entropy(p0, p1)`: $H_{\mathrm{Shannon}} = -p_0\log_2 p_0 - p_1\log_2 p_1$. Terms with $p = 0$ are treated as $0 \log_2 0 = 0$.
- `min_entropy(p0, p1)`: $H_{\min} = -\log_2 \max(p_0, p_1)$
- `evaluate(counts)`: computes $n_0, n_1, \hat{P}(0), \hat{P}(1)$, bias, $H_{\mathrm{Shannon}}$, and $H_{\min}$ together from the measurement-count dictionary `counts`
- `expected_bias(shots)`: the bias $\sqrt{2/\pi}\,/\,(2\sqrt{N})$ expected for ideal fair bits. Used as a reference for comparison with the results
- `show_metrics(r)`: displays P(0), P(1), bias, and entropies, with reference values for fair bits (used in 4.4 and 5.6)
- `show_table(rows, first_col)`: displays the results as a table

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `shannon_entropy` | Shannon エントロピー（float） | `shannon_entropy(0.5, 0.5)` |
| `min_entropy` | min-entropy（float） | `min_entropy(0.5, 0.5)` |
| `evaluate` | 測定回数・counts・確率・bias・エントロピーを含む辞書 | `evaluate({"0": 503, "1": 497})` |
| `expected_bias` | 公平なビットの bias の期待値の近似（float） | `expected_bias(100_000)` |
| `show_metrics` | ビット列の指標と公平なビットの目安を表示。戻り値は `None` | `show_metrics(evaluate({"0": 503, "1": 497}))` |
| `show_table` | Markdown の表を表示。戻り値は `None` | `show_table([("H", evaluate({"0": 503, "1": 497}))], "gate")` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `shannon_entropy` | Shannon entropy (float) | `shannon_entropy(0.5, 0.5)` |
| `min_entropy` | Min-entropy (float) | `min_entropy(0.5, 0.5)` |
| `evaluate` | Dictionary containing the number of measurements, counts, probabilities, bias, and entropies | `evaluate({"0": 503, "1": 497})` |
| `expected_bias` | Approximate expected bias of fair bits (float) | `expected_bias(100_000)` |
| `show_metrics` | Displays the metrics of the bit string and reference values for fair bits. Returns `None` | `show_metrics(evaluate({"0": 503, "1": 497}))` |
| `show_table` | Displays a Markdown table. Returns `None` | `show_table([("H", evaluate({"0": 503, "1": 497}))], "gate")` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>shannon_entropy</code></td><td><code>p0</code></td><td>0 が出る確率。0〜1 の数値（例：<code>0.5</code>）</td><td>必須</td></tr>
<tr><td><code>p1</code></td><td>1 が出る確率。<code>p0 + p1 = 1</code> にする（例：<code>0.5</code>）</td><td>必須</td></tr>
<tr><td rowspan="2"><code>min_entropy</code></td><td><code>p0</code></td><td>0 が出る確率。0〜1 の数値（例：<code>0.5</code>）</td><td>必須</td></tr>
<tr><td><code>p1</code></td><td>1 が出る確率。<code>p0 + p1 = 1</code> にする（例：<code>0.5</code>）</td><td>必須</td></tr>
<tr><td><code>evaluate</code></td><td><code>counts</code></td><td>1 ビットの測定集計の辞書。キーは文字列 <code>"0"</code>・<code>"1"</code>、値は非負の整数。合計は 1 以上。出なかったキーは省略可</td><td>必須</td></tr>
<tr><td><code>expected_bias</code></td><td><code>shots</code></td><td>測定回数。1 以上の整数（例：<code>100_000</code>）</td><td>必須</td></tr>
<tr><td><code>show_metrics</code></td><td><code>r</code></td><td><code>evaluate</code> の戻り値</td><td>必須</td></tr>
<tr><td rowspan="2"><code>show_table</code></td><td><code>rows</code></td><td><code>(ラベル, evaluate の戻り値)</code> のリスト。例：<code>[("H", r)]</code></td><td>必須</td></tr>
<tr><td><code>first_col</code></td><td>表の先頭列の見出し文字列。例：<code>"gate"</code>、<code>"trial"</code></td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>shannon_entropy</code></td><td><code>p0</code></td><td>Probability of getting 0. A number from 0 to 1 (e.g., <code>0.5</code>)</td><td>Required</td></tr>
<tr><td><code>p1</code></td><td>Probability of getting 1. Must satisfy <code>p0 + p1 = 1</code> (e.g., <code>0.5</code>)</td><td>Required</td></tr>
<tr><td rowspan="2"><code>min_entropy</code></td><td><code>p0</code></td><td>Probability of getting 0. A number from 0 to 1 (e.g., <code>0.5</code>)</td><td>Required</td></tr>
<tr><td><code>p1</code></td><td>Probability of getting 1. Must satisfy <code>p0 + p1 = 1</code> (e.g., <code>0.5</code>)</td><td>Required</td></tr>
<tr><td><code>evaluate</code></td><td><code>counts</code></td><td>Dictionary of 1-bit measurement tallies. Keys are the strings <code>"0"</code> and <code>"1"</code>; values are non-negative integers with a total of at least 1. Keys that did not occur may be omitted</td><td>Required</td></tr>
<tr><td><code>expected_bias</code></td><td><code>shots</code></td><td>Number of measurements. An integer of at least 1 (e.g., <code>100_000</code>)</td><td>Required</td></tr>
<tr><td><code>show_metrics</code></td><td><code>r</code></td><td>Return value of <code>evaluate</code></td><td>Required</td></tr>
<tr><td rowspan="2"><code>show_table</code></td><td><code>rows</code></td><td>List of <code>(label, return value of evaluate)</code>. E.g., <code>[("H", r)]</code></td><td>Required</td></tr>
<tr><td><code>first_col</code></td><td>Heading string for the first column of the table. E.g., <code>"gate"</code>, <code>"trial"</code></td><td>Required</td></tr>
</tbody>
</table>


In [ ]:
def shannon_entropy(p0, p1):
    # Treat 0 log 0 as 0
    return -sum(p * math.log2(p) for p in (p0, p1) if p > 0)


def min_entropy(p0, p1):
    return -math.log2(max(p0, p1))


def evaluate(counts):
    n0 = counts.get("0", 0)
    n1 = counts.get("1", 0)
    shots = n0 + n1
    p0 = n0 / shots
    p1 = n1 / shots
    return {
        "shots": shots,
        "n0": n0,
        "n1": n1,
        "P0": p0,
        "P1": p1,
        "bias": abs(p1 - 1 / 2),
        "shannon": shannon_entropy(p0, p1),
        "min_entropy": min_entropy(p0, p1),
    }


def expected_bias(shots):
    # Expected bias when a fair bit is measured shots times (normal approximation)
    return math.sqrt(2 / math.pi) / (2 * math.sqrt(shots))


def show_table(rows, first_col):
    # rows: list of (label, result of evaluate)
    lines = [
        f"| {first_col} | counts (0) | counts (1) | P(0) | P(1) | bias | Shannon entropy | min-entropy |",
        "|---|---|---|---|---|---|---|---|",
    ]
    for label, r in rows:
        lines.append(
            f"| {label} | {r['n0']} | {r['n1']} | {r['P0']:.4f} | {r['P1']:.4f} "
            f"| {r['bias']:.5f} | {r['shannon']:.5f} | {r['min_entropy']:.5f} |"
        )
    display(Markdown("\n".join(lines)))


def show_metrics(r):
    # Metrics of the bit string, with reference values for fair bits
    for label, k in [("P(0)", "P0"), ("P(1)", "P1"), ("bias", "bias"),
                     ("Shannon entropy", "shannon"), ("min-entropy", "min_entropy")]:
        print(f"{label:15} = {r[k]:.6f}")
    b_ref = expected_bias(r["shots"])
    print(f"[Reference] expected bias for fair bits = {b_ref:.6f}")
    print(f"[Reference] min-entropy at that bias    = {-math.log2(1 / 2 + b_ref):.6f}")

<!-- ## 3. C1 回路を作る -->

## 3. Building the C1 circuit

<!-- 説明パート A・B の C1 回路を作る関数 `build_c1(gate)` を定義します。 -->

We define the function `build_c1(gate)`, which builds the C1 circuit of Explanation Parts A and B.

<!--
1. `QuantumCircuit(1, 1)` で量子ビット 1 個・古典ビット 1 個の回路を作る（量子ビットは $|0\rangle$ から始まる）
2. 選んだゲート（$H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$ のいずれか）を量子ビット 0 に作用させる
3. 量子ビット 0 を測定し、結果を古典ビット 0 に記録する
-->

1. Create a circuit with 1 qubit and 1 classical bit using `QuantumCircuit(1, 1)` (the qubit starts in $|0\rangle$)
2. Apply the chosen gate (one of $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$) to qubit 0
3. Measure qubit 0 and record the result in classical bit 0

<!-- 3 つのゲートすべてについて、元の回路と transpile 後の回路を表示します。$H_{\mathrm{Gate}}$ だけが $R_Y(\pi/2)$ と $R_X(\pi)$ の 2 個に分解され、`global phase: π/2` と表示されることを確認してください（説明パート D）。最後に `Operator(...).equiv(...)` で、変換の前後で回路が同じ演算になっていることを確かめます。 -->

For all three gates, we display the original circuit and the transpiled circuit. Confirm that only $H_{\mathrm{Gate}}$ is decomposed into the two gates $R_Y(\pi/2)$ and $R_X(\pi)$, and that `global phase: π/2` is displayed (Explanation Part D). Finally, `Operator(...).equiv(...)` confirms that the circuit implements the same operation before and after the conversion.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `build_c1` | 測定を含む 1 量子ビット回路（QuantumCircuit） | `build_c1("H")` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `build_c1` | 1-qubit circuit including measurement (QuantumCircuit) | `build_c1("H")` |

<br>

<!--
| 関数 | 引数 | 入力する値 | 省略時 |
|---|---|---|---|
| `build_c1` | `gate` | 大文字小文字を区別して `"H"`・`"Rx"`・`"Ry"` のいずれか | 必須 |
-->

| Function | Argument | Value to pass | If omitted |
|---|---|---|---|
| `build_c1` | `gate` | One of `"H"`, `"Rx"`, `"Ry"` (case-sensitive) | Required |


In [ ]:
def build_c1(gate):
    if gate not in GATES:
        raise ValueError(f"gate must be one of {GATES} (got: {gate!r})")

    qc = QuantumCircuit(1, 1)
    if gate == "H":
        qc.h(0)
    elif gate == "Rx":
        qc.rx(math.pi / 2, 0)
    else:  # "Ry"
        qc.ry(math.pi / 2, 0)
    qc.measure(0, 0)
    return qc


for gate in GATES:
    qc = build_c1(gate)
    qc_t = transpile(qc, basis_gates=BASIS_GATES)

    print(f"===== {gate} =====")
    print("[Original circuit]")
    print(qc.draw())
    print("[Transpiled circuit]")
    print(qc_t.draw())

    # Check, excluding measurements, that the operation is the same before and after conversion
    same = Operator(qc.remove_final_measurements(inplace=False)).equiv(
        Operator(qc_t.remove_final_measurements(inplace=False))
    )
    print("Same operation before and after conversion:", same)
    print()

<!-- ## 4. ノイズなしのシミュレーション -->

## 4. Noiseless simulation

<!-- まずノイズのない理想的なシミュレータで C1 回路を測定します。理想的には $P(0) = P(1) = 1/2$ ですが、有限回の測定では統計的に揺らぎます（説明パート C）。 -->

First, we measure the C1 circuit on an ideal simulator with no noise. Ideally $P(0) = P(1) = 1/2$, but a finite number of measurements fluctuates statistically (Explanation Part C).

<!-- ### 4.1 ゲートと測定回数を選ぶ -->

### 4.1 Choosing the gate and the number of measurements

<!-- `GATE_IDEAL` に `"H"`, `"Rx"`, `"Ry"` のいずれかを、`SHOTS_IDEAL` に測定回数を指定します。 -->

Set `GATE_IDEAL` to one of `"H"`, `"Rx"`, `"Ry"`, and set `SHOTS_IDEAL` to the number of measurements.

In [ ]:
GATE_IDEAL = "H"           # Choose from "H", "Rx", "Ry"
SHOTS_IDEAL = 1_000_000    # Number of measurements

<!-- ### 4.2 回路を作って transpile する -->

### 4.2 Building and transpiling the circuit

<!-- `build_c1` で回路を作り、`transpile` でネイティブゲート `BASIS_GATES` だけを使う回路に変換します。 -->

We build the circuit with `build_c1` and use `transpile` to convert it into a circuit that uses only the native gates `BASIS_GATES`.

In [ ]:
qc_ideal = build_c1(GATE_IDEAL)
qc_ideal_t = transpile(qc_ideal, basis_gates=BASIS_GATES)

print(qc_ideal_t.draw())

<!-- ### 4.3 測定する -->

### 4.3 Measuring

<!-- `AerSimulator()` をノイズなしで作り、`backend.run(..., shots=SHOTS_IDEAL, memory=True)` で測定します。 -->

We create a noiseless `AerSimulator()` and measure with `backend.run(..., shots=SHOTS_IDEAL, memory=True)`.

<!--
- `result.get_counts()`：0 と 1 が出た回数
- `result.get_memory()`：各 shot の結果を測定順に並べたリスト。実機なら乱数の元になる**生のビット列（raw data）**に相当する模擬データです。長いので先頭 50 個だけ表示します。シミュレータの結果は古典コンピュータの擬似乱数から作られ、シードで再現できるので、真の量子乱数ではなく、暗号には使えません。
-->

- `result.get_counts()`: the number of times 0 and 1 occurred
- `result.get_memory()`: a list of the result of each shot in measurement order. This is simulated data corresponding to the **raw bit string (raw data)** that would be the source of random numbers on real hardware. Since it is long, only the first 50 entries are displayed. Simulator results are generated from a classical computer's pseudorandom numbers and are reproducible with a seed, so they are not true quantum random numbers and must not be used for cryptography.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、このセルの raw dataを `4.3_ideal_single/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., the raw data of this cell is saved in `4.3_ideal_single/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
backend_ideal = AerSimulator(seed_simulator=SEED)

job = backend_ideal.run(qc_ideal_t, shots=SHOTS_IDEAL, memory=True)
result = job.result()

counts_ideal = result.get_counts()
raw_bits_ideal = result.get_memory()

print("counts =", counts_ideal)
print("raw bits (first 50) =", "".join(raw_bits_ideal[:50]))
print("Length of raw bits =", len(raw_bits_ideal))

# Save the raw data (only when SAVE_RESULTS = True)
if SAVE_RESULTS:
    path = save_raw(output_dir("4.3_ideal_single") / f"c1_{GATE_IDEAL}_ideal.txt", raw_bits_ideal,
                    GATE_IDEAL, SHOTS_IDEAL, SEED, metadata={"condition": "ideal"})
    print("Saved:", path)

<!-- ### 4.4 評価指標を計算する -->

### 4.4 Computing the evaluation metrics

<!-- `evaluate` で $\hat{P}(0)$, $\hat{P}(1)$, bias, Shannon エントロピー, min-entropy を計算します。あわせて、公平なビットで期待される bias の目安 $\sqrt{2/\pi}\,/\,(2\sqrt{N})$ と、それに対応する min-entropy $-\log_2(1/2 + \mathrm{bias})$ も表示します。観測された bias がこの目安と同じくらいなら、ずれは統計的な揺らぎで説明できます。 -->

`evaluate` computes $\hat{P}(0)$, $\hat{P}(1)$, the bias, the Shannon entropy, and the min-entropy. We also display the reference bias expected for fair bits, $\sqrt{2/\pi}\,/\,(2\sqrt{N})$, and the corresponding min-entropy $-\log_2(1/2 + \mathrm{bias})$. If the observed bias is about the same as this reference, the deviation can be explained by statistical fluctuation.

In [ ]:
r = evaluate(counts_ideal)
show_metrics(r)

<!-- ### 4.5 測定を繰り返す -->

### 4.5 Repeating the measurement

<!-- 1 回の結果だけでは、偏りが回路のせいなのか偶然なのか分かりません。そこで同じ測定を `N_TRIALS` 回繰り返し、ばらつきを見ます。以降でも使うので、「回路を作る → transpile → 測定 → 評価」を関数 `run_c1` にまとめます。繰り返しでは生のビット列は使わないので、`memory=True` は付けません。 -->

A single result cannot tell us whether a bias is due to the circuit or to chance. So we repeat the same measurement `N_TRIALS` times and look at the spread. Because it is also used later, we package "build the circuit → transpile → measure → evaluate" into the function `run_c1`. The repetitions do not use the raw bit string, so `memory=True` is not set.

<!-- `run_c1` に `raw_path` を指定すると、測定順のビット列を別ファイルに保存します。 -->

If `raw_path` is passed to `run_c1`, the bit string in measurement order is saved to a separate file.

```python
r = run_c1("H", 100_000, seed=2026, raw_path="results/c1/manual/c1_H_ideal.txt")
```

<!--
`c1_H_ideal.txt` は 1 shot を 1 文字とした 0/1 の列（改行なし）です。
隣の `c1_H_ideal.txt.json` に測定条件・ノイズモデル・counts・評価結果を保存します。
既存ファイルがある場合は上書きせずエラーにします。保存先を省略すると、集計結果だけを返します。
繰り返し測定は `run_trials(..., raw_dir="results/c1/manual/trials")` で試行ごとに保存できます。
-->

`c1_H_ideal.txt` is a sequence of 0s and 1s with one character per shot (no line breaks).
The adjacent file `c1_H_ideal.txt.json` stores the measurement conditions, noise model, counts, and evaluation results.
If a file already exists, an error is raised rather than overwriting it. If the destination is omitted, only the aggregated results are returned.
For repeated measurements, `run_trials(..., raw_dir="results/c1/manual/trials")` saves the data for each trial.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、次のセルの最後の `run_trials` に `raw_dir` として `RUN_DIR/4.5_ideal_trials` を渡すので、試行ごとのファイル（`c1_H_ideal_trial001.txt` など）と、全試行の一覧 `trials.csv` を保存します。上の例のように自分で `raw_path` や `raw_dir` を指定した場合は、`SAVE_RESULTS` に関係なくその場所に保存します。 -->

If `SAVE_RESULTS = True` in 1., the `run_trials` call at the end of the next cell is given `RUN_DIR/4.5_ideal_trials` as `raw_dir`, so one file per trial (`c1_H_ideal_trial001.txt` and so on) and the list of all trials, `trials.csv`, are saved. If you give your own `raw_path` or `raw_dir` as in the examples above, the files are saved there regardless of `SAVE_RESULTS`.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `run_c1` | `evaluate` と同じ評価結果の辞書。保存時も戻り値は同じ | `run_c1("H", 100_000, seed=2026, raw_path="results/c1/manual/c1_H_ideal.txt")` |
| `run_trials` | `(試行番号, 評価結果の辞書)` のリスト。`raw_dir` を指定すると、試行ごとのファイルと `trials.csv` を保存する | `run_trials("H", 100_000, 3, seed=2026, raw_dir="results/c1/manual/trials")` |
| `run_conditions` | `(ラベル, 評価結果の辞書)` のリスト。`(ゲート, 条件の名前, NoiseModel)` の組を 1 回ずつ測定する（4.6・5.8 で使う）。`SAVE_RESULTS = True` なら raw data と `summary.csv` を保存する | `run_conditions([("H", "ideal", None)], 1_000, "4.6_ideal_gates")` |
| `offset_seed` | `seed + k`（`seed` が `None` なら `None`）。測定ごとに違うシードにする | `offset_seed(2026, 3)` |
| `summarize` | bias・min-entropy の要約を表示。戻り値は `None` | `summarize(rows_ideal)` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `run_c1` | Evaluation dictionary, same as `evaluate`. The return value is the same when saving | `run_c1("H", 100_000, seed=2026, raw_path="results/c1/manual/c1_H_ideal.txt")` |
| `run_trials` | List of `(trial number, evaluation dictionary)`. If `raw_dir` is given, one file per trial and `trials.csv` are saved | `run_trials("H", 100_000, 3, seed=2026, raw_dir="results/c1/manual/trials")` |
| `run_conditions` | List of `(label, evaluation dictionary)`. Measures each `(gate, condition name, NoiseModel)` once (used in 4.6 and 5.8). With `SAVE_RESULTS = True`, the raw data and `summary.csv` are saved | `run_conditions([("H", "ideal", None)], 1_000, "4.6_ideal_gates")` |
| `offset_seed` | `seed + k` (`None` if `seed` is `None`). Gives each measurement a different seed | `offset_seed(2026, 3)` |
| `summarize` | Displays a summary of bias and min-entropy. Returns `None` | `summarize(rows_ideal)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="8"><code>run_c1</code></td><td><code>gate</code></td><td><code>"H"</code>・<code>"Rx"</code>・<code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td><code>shots</code></td><td>測定回数。1 以上の整数（例：<code>100_000</code>）</td><td>必須</td></tr>
<tr><td><code>noise_model</code></td><td><code>build_noise_model("both")</code> などで作った NoiseModel。ノイズなしなら <code>None</code></td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>再現用の非負整数（例：<code>2026</code>）。毎回異なる乱数なら <code>None</code></td><td><code>None</code></td></tr>
<tr><td><code>raw_path</code></td><td>ビット列の保存先の文字列または Path（例：<code>"results/c1/manual/c1_H_ideal.txt"</code>）。<code>None</code> なら保存しない。未使用のファイル名を指定</td><td><code>None</code></td></tr>
<tr><td><code>method</code></td><td>通常は <code>"automatic"</code>。密度行列方式を指定する場合は <code>"density_matrix"</code></td><td><code>'automatic'</code></td></tr>
<tr><td><code>optimization_level</code></td><td>Qiskit の回路変換時の最適化レベル。<code>0</code>・<code>1</code>・<code>2</code>・<code>3</code>、既定値を使うなら <code>None</code>。5.9 では、最適化でゲートの順序や個数が変わらないよう <code>0</code> に固定</td><td><code>None</code></td></tr>
<tr><td><code>metadata</code></td><td>JSON に保存できる追加情報の辞書（例：<code>{"trial": 1}</code>）。不要なら <code>None</code>。<code>raw_path</code> 指定時に保存</td><td><code>None</code></td></tr>
<tr><td rowspan="7"><code>run_trials</code></td><td><code>gate</code></td><td><code>"H"</code>・<code>"Rx"</code>・<code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td><code>shots</code></td><td>1 試行あたりの測定回数。1 以上の整数</td><td>必須</td></tr>
<tr><td><code>n_trials</code></td><td>繰り返す試行数。1 以上の整数（例：<code>3</code>）</td><td>必須</td></tr>
<tr><td><code>noise_model</code></td><td>NoiseModel。ノイズなしなら <code>None</code></td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>非負整数または <code>None</code>。整数の場合は試行番号を加えたシードで実行</td><td><code>None</code></td></tr>
<tr><td><code>raw_dir</code></td><td>試行別のビット列と JSON を保存するフォルダ。文字列または Path。<code>None</code> なら保存しない。既存ファイルは上書きしない</td><td><code>None</code></td></tr>
<tr><td><code>label</code></td><td>ファイル名に入れる条件の名前（例：<code>"ideal"</code>、<code>"noise-both"</code>）。<code>trials.csv</code> の <code>condition</code> 列にもなる</td><td><code>"ideal"</code></td></tr>
<tr><td rowspan="3"><code>run_conditions</code></td><td><code>conditions</code></td><td><code>(ゲート, 条件の名前, NoiseModel または None)</code> のリスト。条件の名前はファイル名に入る</td><td>必須</td></tr>
<tr><td><code>shots</code></td><td>1 回あたりの測定回数</td><td>必須</td></tr>
<tr><td><code>folder</code></td><td><code>RUN_DIR</code> の中の保存先のフォルダ名（例：<code>"4.6_ideal_gates"</code>）</td><td>必須</td></tr>
<tr><td rowspan="2"><code>offset_seed</code></td><td><code>seed</code></td><td>非負整数または <code>None</code></td><td>必須</td></tr>
<tr><td><code>k</code></td><td>足す数（試行番号や条件の番号）</td><td>必須</td></tr>
<tr><td><code>summarize</code></td><td><code>rows</code></td><td><code>run_trials</code> が返す <code>(試行番号, 評価結果)</code> のリスト。少なくとも 1 試行が必要</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="8"><code>run_c1</code></td><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td><code>shots</code></td><td>Number of measurements. An integer of at least 1 (e.g., <code>100_000</code>)</td><td>Required</td></tr>
<tr><td><code>noise_model</code></td><td>A NoiseModel created with, e.g., <code>build_noise_model("both")</code>. <code>None</code> for no noise</td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>Non-negative integer for reproducibility (e.g., <code>2026</code>). <code>None</code> for different random numbers each time</td><td><code>None</code></td></tr>
<tr><td><code>raw_path</code></td><td>Destination for the bit string, as a string or Path (e.g., <code>"results/c1/manual/c1_H_ideal.txt"</code>). <code>None</code> means do not save. Specify an unused file name</td><td><code>None</code></td></tr>
<tr><td><code>method</code></td><td>Normally <code>"automatic"</code>. Use <code>"density_matrix"</code> to select the density-matrix method</td><td><code>'automatic'</code></td></tr>
<tr><td><code>optimization_level</code></td><td>Optimization level for Qiskit's circuit conversion. <code>0</code>, <code>1</code>, <code>2</code>, <code>3</code>, or <code>None</code> to use the default. In 5.9 it is fixed at <code>0</code> so that optimization does not change the order or number of gates</td><td><code>None</code></td></tr>
<tr><td><code>metadata</code></td><td>Dictionary of extra information that can be saved as JSON (e.g., <code>{"trial": 1}</code>). <code>None</code> if not needed. Saved when <code>raw_path</code> is specified</td><td><code>None</code></td></tr>
<tr><td rowspan="7"><code>run_trials</code></td><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td><code>shots</code></td><td>Number of measurements per trial. An integer of at least 1</td><td>Required</td></tr>
<tr><td><code>n_trials</code></td><td>Number of trials to repeat. An integer of at least 1 (e.g., <code>3</code>)</td><td>Required</td></tr>
<tr><td><code>noise_model</code></td><td>A NoiseModel. <code>None</code> for no noise</td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>Non-negative integer or <code>None</code>. If an integer, each trial runs with the seed plus the trial number</td><td><code>None</code></td></tr>
<tr><td><code>raw_dir</code></td><td>Folder in which to save the bit string and JSON of each trial, as a string or Path. <code>None</code> means do not save. Existing files are not overwritten</td><td><code>None</code></td></tr>
<tr><td><code>label</code></td><td>Name of the condition put in the file names (e.g., <code>"ideal"</code>, <code>"noise-both"</code>). Also used as the <code>condition</code> column of <code>trials.csv</code></td><td><code>"ideal"</code></td></tr>
<tr><td rowspan="3"><code>run_conditions</code></td><td><code>conditions</code></td><td>List of <code>(gate, condition name, NoiseModel or None)</code>. The condition name goes into the file name</td><td>Required</td></tr>
<tr><td><code>shots</code></td><td>Number of measurements per condition</td><td>Required</td></tr>
<tr><td><code>folder</code></td><td>Name of the destination folder in <code>RUN_DIR</code> (e.g., <code>"4.6_ideal_gates"</code>)</td><td>Required</td></tr>
<tr><td rowspan="2"><code>offset_seed</code></td><td><code>seed</code></td><td>Non-negative integer or <code>None</code></td><td>Required</td></tr>
<tr><td><code>k</code></td><td>Number to add (trial number or condition number)</td><td>Required</td></tr>
<tr><td><code>summarize</code></td><td><code>rows</code></td><td>List of <code>(trial number, evaluation result)</code> returned by <code>run_trials</code>. At least one trial is required</td><td>Required</td></tr>
</tbody>
</table>

<br>

<!-- `run_c1` の `method`・`optimization_level`・`metadata` は、`method="density_matrix"` のように引数名を付けて指定します。保存する `.txt` は測定順の 0/1 の列、`.txt.json` は条件と集計結果です。 -->

Specify `method`, `optimization_level`, and `metadata` of `run_c1` by keyword, as in `method="density_matrix"`. The saved `.txt` is the sequence of 0s and 1s in measurement order, and the `.txt.json` holds the conditions and aggregated results.


In [ ]:
N_TRIALS = 10


def offset_seed(seed, k):
    # A different seed for each run: seed + k (None stays None, i.e., different results each time)
    return None if seed is None else seed + k


def run_c1(gate, shots, noise_model=None, seed=None, raw_path=None,
           *, method="automatic", optimization_level=None, metadata=None):
    # Build the circuit -> transpile -> measure -> evaluate. With raw_path, the raw data is also saved.
    if raw_path is not None:
        ensure_new(raw_path)   # stop before measuring if the files already exist
    qc_t = transpile(build_c1(gate), basis_gates=BASIS_GATES, optimization_level=optimization_level)
    backend = AerSimulator(method=method, noise_model=noise_model, seed_simulator=seed)
    result = backend.run(qc_t, shots=shots, memory=raw_path is not None).result()
    if raw_path is not None:
        save_raw(raw_path, result.get_memory(), gate, shots, seed, noise_model,
                 method=method, optimization_level=optimization_level, metadata=metadata)
    return evaluate(result.get_counts())


def run_trials(gate, shots, n_trials, noise_model=None, seed=None, raw_dir=None, label="ideal"):
    # Repeat the same measurement. With raw_dir, one file per trial and trials.csv are saved.
    rows, records = [], []
    for trial in range(1, n_trials + 1):
        name = f"c1_{gate}_{label}_trial{trial:03d}.txt"
        raw_path = None if raw_dir is None else Path(raw_dir) / name
        s = offset_seed(seed, trial)
        r = run_c1(gate, shots, noise_model, s, raw_path, metadata={"condition": label, "trial": trial})
        rows.append((trial, r))
        records.append(dict(file=name, gate=gate, condition=label, trial=trial, seed=s, **r))
    if raw_dir is not None:
        write_csv(Path(raw_dir) / "trials.csv", records)   # written after all trials have finished
    return rows


def run_conditions(conditions, shots, folder):
    # Measure each (gate, condition, noise_model) once (used in 4.6 and 5.8).
    # With SAVE_RESULTS = True, the raw data and summary.csv are saved in output_dir(folder).
    out = output_dir(folder)
    rows, records = [], []
    for k, (gate, condition, noise_model) in enumerate(conditions):
        name = f"c1_{gate}_{condition}.txt"
        raw_path = None if out is None else out / name
        s = offset_seed(SEED, k)   # the same seed would align the results of different conditions
        r = run_c1(gate, shots, noise_model, s, raw_path, metadata={"condition": condition})
        rows.append((f"{gate} / {condition}", r))
        records.append(dict(file=name, gate=gate, condition=condition, seed=s, **r))
    if out is not None:
        write_csv(out / "summary.csv", records)
    return rows


def summarize(rows):
    biases = [r["bias"] for _, r in rows]
    mins = [r["min_entropy"] for _, r in rows]
    mean = lambda xs: sum(xs) / len(xs)
    print(f"mean bias        = {mean(biases):.6f} (reference {expected_bias(rows[0][1]['shots']):.6f})")
    print(f"mean min-entropy = {mean(mins):.6f} (min {min(mins):.6f}, max {max(mins):.6f})")


rows_ideal = run_trials(GATE_IDEAL, SHOTS_IDEAL, N_TRIALS, seed=SEED, raw_dir=output_dir("4.5_ideal_trials"))
show_table(rows_ideal, "trial")
summarize(rows_ideal)

<!-- ### 4.6 （参考）3 つのゲートを比べる -->

### 4.6 (Optional) Comparing the three gates

<!-- 同じ測定回数で $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$ を 1 回ずつ測定して並べます。ノイズがないので、3 つの違いは統計的な揺らぎだけのはずです。 -->

We measure $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, and $R_Y(\pi/2)$ once each with the same number of measurements and list the results side by side. Since there is no noise, the only differences among the three should be statistical fluctuations.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、3 つのゲートの raw data と一覧 `summary.csv`を `4.6_ideal_gates/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., the raw data for the three gates and their list, `summary.csv`, are saved in `4.6_ideal_gates/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
rows = run_conditions([(gate, "ideal", None) for gate in GATES], SHOTS_IDEAL, "4.6_ideal_gates")
show_table(rows, "gate / condition")

<!-- ## 5. ノイズありのシミュレーション -->

## 5. Noisy simulation

<!-- 次に、説明パート E のノイズを加えて測定します。 -->

Next, we add the noise of Explanation Part E and measure.

<!-- ### 5.1 ノイズのパラメータ -->

### 5.1 Noise parameters

<!-- 論文の Table II の値を設定します（$\lambda$ は説明パート E のとおり、平均ゲート忠実度が 99.9% になる値にします）。熱緩和誤差の関数には $T_1$, $T_2$, ゲート時間を**同じ単位**で渡す必要があるので、すべてナノ秒（ns）にそろえます。 -->

We set the values from Table II of the paper ($\lambda$ is the value that gives an average gate fidelity of 99.9%, as in Explanation Part E). The thermal relaxation error function requires $T_1$, $T_2$, and the gate time in the **same unit**, so everything is expressed in nanoseconds (ns).

In [ ]:
DEPOLARIZING_PARAM = 0.00198  # Depolarizing parameter λ (gives an average gate fidelity of 99.9% including thermal relaxation)
READOUT_ERROR_RATE = 0.03     # Readout error (fidelity 97%)

T1 = 0.964e6                  # Relaxation time 0.964 ms = 0.964e6 ns
T2 = 1.155e6                  # Dephasing time 1.155 ms = 1.155e6 ns
GATE_TIME = 20                # 1-qubit gate time 20 ns

print(f"Probability that |1> decays to |0> during the gate time ≈ {1 - math.exp(-GATE_TIME / T1):.2e}")

<!-- ### 5.2 ノイズモデルを作る関数 -->

### 5.2 Function for building the noise model

<!-- ノイズモデルは、ノイズの**成分**を組み合わせて作ります。次のセルの 3 つの部分でできています。 -->

The noise model is built by combining noise **components**. The next cell has three parts.

<!--
- `NOISE_COMPONENTS`：ノイズの成分の一覧。成分ごとに、強さから Qiskit Aer の誤差を作る関数を持ちます。ゲートの後に加える量子的な誤差は `gate_errors`、測定の読み出し誤差は `readout` に書きます。`gate_errors` のキーはゲートの名前（`"rx"`, `"ry"`）です。
- `NOISE_KINDS`：5.3〜5.8 で使うノイズの種類。名前と、`{成分: 強さ}` の組です。`"gate"` は脱分極 + 熱緩和、`"readout"` は対称な読み出し誤差、`"both"` はその両方で、強さは 5.1 の定数です。
- `build_noise_model(kind)`：`NOISE_KINDS` の名前、または `{成分: 強さ}` の辞書から `NoiseModel` を作ります。同じゲートの誤差は書いた順に `compose` で合成し（脱分極のあとに熱緩和）、読み出し誤差は混同行列を掛け合わせて、それぞれ 1 回だけ加えます。Qiskit Aer では、読み出し誤差を 2 回加えると後のもので上書きされるためです。5.9 の `build_sweep_noise` も同じ関数を使います。
-->

- `NOISE_COMPONENTS`: the list of noise components. Each component has functions that create Qiskit Aer errors from a strength. Quantum errors added after gates go in `gate_errors`, and readout errors at measurement go in `readout`. The keys of `gate_errors` are gate names (`"rx"`, `"ry"`).
- `NOISE_KINDS`: the noise kinds used in 5.3–5.8. Each is a name and a set of `{component: strength}`. `"gate"` is depolarizing + thermal relaxation, `"readout"` is a symmetric readout error, and `"both"` is both; the strengths are the constants in 5.1.
- `build_noise_model(kind)`: builds a `NoiseModel` from a name in `NOISE_KINDS` or from a dictionary `{component: strength}`. Errors on the same gate are composed with `compose` in the order written (depolarizing, then thermal relaxation), readout errors are combined by multiplying their confusion matrices, and each is added only once, because in Qiskit Aer adding a second readout error replaces the first. `build_sweep_noise` in 5.9 uses the same function.

<!-- このノートブックで定義している成分は次のとおりです。 -->

The components defined in this notebook are as follows.

<!--
| 成分 | 加える誤差 | 強さ |
|---|---|---|
| `depolarizing` | `rx`, `ry` の後に `depolarizing_error(λ, 1)` | λ（0〜1） |
| `thermal` | `rx`, `ry` の後に `thermal_relaxation_error(T1, T2, 倍率 × GATE_TIME)` | ゲート時間の倍率（1 が 5.1 の値） |
| `readout` | 対称な読み出し誤差の混同行列 $\begin{pmatrix} 1-\varepsilon & \varepsilon \\ \varepsilon & 1-\varepsilon \end{pmatrix}$ | ε（0〜1） |
-->

| Component | Error added | Strength |
|---|---|---|
| `depolarizing` | `depolarizing_error(λ, 1)` after `rx`, `ry` | λ (0–1) |
| `thermal` | `thermal_relaxation_error(T1, T2, multiplier × GATE_TIME)` after `rx`, `ry` | Gate-time multiplier (1 is the value in 5.1) |
| `readout` | Confusion matrix of a symmetric readout error, $\begin{pmatrix} 1-\varepsilon & \varepsilon \\ \varepsilon & 1-\varepsilon \end{pmatrix}$ | ε (0–1) |

<!-- **ノイズの種類を増やす手順** -->

**How to add a kind of noise**

<!--
1. `NOISE_COMPONENTS` に成分を 1 つ加えます（下の例と、その下の Qiskit Aer の誤差の表を参照）。
2. 5.3〜5.8 で使うときは、`NOISE_KINDS` に名前と `{成分: 強さ}` を加え、5.3 の `NOISE_KIND` にその名前を指定します。5.8 の比較と、保存するファイルの名前（`noise-<名前>`）は自動で増えます。
3. 5.9 で強さを変えるときは、5.9 の `SWEEP_VALUES` に成分の名前と強さのリストを加えます。グラフの列、CSV、保存のフォルダは自動で増えます。
-->

1. Add one component to `NOISE_COMPONENTS` (see the example below and the table of Qiskit Aer errors after it).
2. To use it in 5.3–5.8, add a name and `{component: strength}` to `NOISE_KINDS`, and set `NOISE_KIND` in 5.3 to that name. The comparison in 5.8 and the names of the saved files (`noise-<name>`) follow automatically.
3. To vary its strength in 5.9, add the component name and a list of strengths to `SWEEP_VALUES` in 5.9. The plot columns, the CSV files, and the saved folders follow automatically.

<!-- 例：エネルギー緩和だけを表す振幅減衰を加える場合（次のセルの後に、新しいセルで実行します） -->

Example: adding amplitude damping, which models energy relaxation only (run in a new cell after the next cell)

```python
from qiskit_aer.noise import amplitude_damping_error

NOISE_COMPONENTS["amplitude_damping"] = dict(
    label="Amplitude damping probability gamma", max=1,
    gate_errors=lambda s: {("rx", "ry"): amplitude_damping_error(s)})
NOISE_KINDS["amp_damping"] = {"amplitude_damping": 0.01}   # NOISE_KIND = "amp_damping" in 5.3
# In the settings cell of 5.9: SWEEP_VALUES["amplitude_damping"] = [i / 50 for i in range(11)]
```

<!-- **Qiskit Aer で加えられるノイズ**（qiskit-aer 0.17） -->

**Noise that Qiskit Aer can add** (qiskit-aer 0.17)

<!-- 量子的な誤差（`QuantumError`）はゲートの後に、古典的な読み出し誤差（`ReadoutError`）は測定のときに加わります。量子的な誤差は `compose`（順に作用）や `tensor`（量子ビットを並べる）で組み合わせられます。 -->

Quantum errors (`QuantumError`) act after gates, and classical readout errors (`ReadoutError`) act at measurement. Quantum errors can be combined with `compose` (applied in sequence) and `tensor` (placed on qubits side by side).

<!--
| 関数（`qiskit_aer.noise`） | 種類 | 主な引数 | 内容・使いどころ |
|---|---|---|---|
| `depolarizing_error(param, num_qubits)` | 量子（ゲートの後） | 脱分極パラメータ λ、量子ビット数 | 確率 λ で状態を最大混合状態に置き換える。ゲートの不忠実度をまとめて表す標準的なモデル |
| `thermal_relaxation_error(t1, t2, time, excited_state_population=0)` | 量子 | $T_1$, $T_2$, 時間 | エネルギー緩和（$\|1\rangle \to \|0\rangle$）と位相緩和。ゲート時間や待ち時間の間の緩和 |
| `amplitude_damping_error(param_amp, excited_state_population=0)` | 量子 | 減衰確率 γ | エネルギー緩和だけ（$\|1\rangle \to \|0\rangle$） |
| `phase_damping_error(param_phase)` | 量子 | 位相減衰の強さ | 位相緩和だけ。重ね合わせの干渉が弱まるが、計算基底での測定確率は変えない |
| `phase_amplitude_damping_error(param_amp, param_phase, excited_state_population=0)` | 量子 | γ と位相減衰の強さ | 上の 2 つを同時に |
| `pauli_error([("X", p), ("I", 1 - p)])` | 量子 | パウリ演算子と確率の組 | ビット反転（`X`）、位相反転（`Z`）、両方（`Y`）など。`"XX"` のように 2 量子ビットの相関した誤差も書ける |
| `coherent_unitary_error(unitary)` | 量子 | ユニタリ行列 | 確率的でない系統誤差。回転角のずれ（過回転）など。例：`RXGate(0.01).to_matrix()` |
| `mixed_unitary_error([(U1, p1), (U2, p2), ...])` | 量子 | ユニタリ行列と確率の組 | 確率で選ばれるユニタリ。パウリ以外の確率的な誤差 |
| `reset_error(prob0, prob1=0)` | 量子 | 確率 | 確率で $\|0\rangle$（または $\|1\rangle$）にリセットされる。状態準備の誤りなど |
| `kraus_error(noise_ops)` | 量子 | Kraus 演算子のリスト | 任意の量子チャネル。上にない誤差を自分で定義する |
| `ReadoutError(probabilities)` | 古典（測定時） | 混同行列 | 測定値の読み間違い。非対称（$P(1\mid 0) \ne P(0\mid 1)$）も表せる |
-->

| Function (`qiskit_aer.noise`) | Type | Main arguments | What it models |
|---|---|---|---|
| `depolarizing_error(param, num_qubits)` | Quantum (after gates) | Depolarizing parameter λ, number of qubits | Replaces the state with the maximally mixed state with probability λ. The standard model summarizing gate infidelity |
| `thermal_relaxation_error(t1, t2, time, excited_state_population=0)` | Quantum | $T_1$, $T_2$, time | Energy relaxation ($\|1\rangle \to \|0\rangle$) and dephasing during a gate time or idle time |
| `amplitude_damping_error(param_amp, excited_state_population=0)` | Quantum | Damping probability γ | Energy relaxation only ($\|1\rangle \to \|0\rangle$) |
| `phase_damping_error(param_phase)` | Quantum | Dephasing strength | Dephasing only. Weakens the interference of superpositions but does not change the measurement probabilities in the computational basis |
| `phase_amplitude_damping_error(param_amp, param_phase, excited_state_population=0)` | Quantum | γ and the dephasing strength | Both of the above at once |
| `pauli_error([("X", p), ("I", 1 - p)])` | Quantum | Pairs of Pauli operator and probability | Bit flip (`X`), phase flip (`Z`), both (`Y`), and so on. Correlated 2-qubit errors such as `"XX"` can also be written |
| `coherent_unitary_error(unitary)` | Quantum | Unitary matrix | A non-random systematic error, such as an over-rotation. Example: `RXGate(0.01).to_matrix()` |
| `mixed_unitary_error([(U1, p1), (U2, p2), ...])` | Quantum | Pairs of unitary and probability | A unitary chosen at random. Random errors other than Paulis |
| `reset_error(prob0, prob1=0)` | Quantum | Probabilities | Resets to $\|0\rangle$ (or $\|1\rangle$) with some probability, e.g., state-preparation errors |
| `kraus_error(noise_ops)` | Quantum | List of Kraus operators | Any quantum channel. For defining an error not listed above |
| `ReadoutError(probabilities)` | Classical (at measurement) | Confusion matrix | Misreading of the measured value. Can be asymmetric ($P(1\mid 0) \ne P(0\mid 1)$) |

<!--
| `NoiseModel` のメソッド | 内容 |
|---|---|
| `add_all_qubit_quantum_error(error, instructions)` | すべての量子ビットの、指定したゲートの後に誤差を加える（このノートブックの `build_noise_model` の方法） |
| `add_quantum_error(error, instructions, qubits)` | 特定の量子ビット（の組）だけに誤差を加える。量子ビットごとに違う誤差や、特定の組の相関した誤差に使う |
| `add_all_qubit_readout_error(error)` / `add_readout_error(error, qubits)` | すべての / 特定の量子ビットの測定に読み出し誤差を加える |
| `NoiseModel.from_backend(backend)` | 実機（または実機を模した backend）のキャリブレーションの値からノイズモデルを作る |
-->

| `NoiseModel` method | What it does |
|---|---|
| `add_all_qubit_quantum_error(error, instructions)` | Adds an error after the specified gates on all qubits (the method used by `build_noise_model` in this notebook) |
| `add_quantum_error(error, instructions, qubits)` | Adds an error only on specific qubits (or pairs of qubits). For errors that differ by qubit, or correlated errors on a specific pair |
| `add_all_qubit_readout_error(error)` / `add_readout_error(error, qubits)` | Adds a readout error to the measurement of all / specific qubits |
| `NoiseModel.from_backend(backend)` | Builds a noise model from the calibration values of a real device (or a backend that mimics one) |

<!--
`NoiseModel` では表せないものもあります。

- **shot どうしの相関**：各 shot は独立にシミュレーションされるので、前の測定結果が次の結果に影響するような相関は作れません。測定後のビット列を加工する必要があります。
- **時間とともに変わるノイズ（ドリフト）**：1 回の実行の間、ノイズモデルは変わりません。測定を分けて、強さを変えながら実行する必要があります。
- **量子ビットの外への漏れ（リーク）**：量子ビットの 2 準位の外の状態は扱えません。
-->

Some things cannot be expressed with `NoiseModel`.

- **Correlations between shots**: each shot is simulated independently, so correlations in which one measurement result affects the next cannot be created. The measured bit string has to be processed afterwards.
- **Noise that changes over time (drift)**: the noise model does not change during one run. The measurement has to be split into parts run with different strengths.
- **Leakage out of the qubit**: states outside the two levels of the qubit cannot be represented.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `build_noise_model` | ゲートの誤差と読み出し誤差を加えた NoiseModel | `build_noise_model("both")`、`build_noise_model({"thermal": 1000})` |
| `noise_components` | 確かめた `{成分: 強さ}` の辞書。知らない成分や範囲外の強さならエラー | `noise_components("both")` |
| `readout_confusion` | 読み出しの成分の混同行列を掛け合わせた 2×2 の numpy 配列。読み出しの成分がなければ `None` | `readout_confusion({"readout": 0.03})` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `build_noise_model` | NoiseModel with gate errors and readout errors | `build_noise_model("both")`, `build_noise_model({"thermal": 1000})` |
| `noise_components` | The checked dictionary `{component: strength}`. An error is raised for an unknown component or a strength out of range | `noise_components("both")` |
| `readout_confusion` | 2×2 numpy array: the product of the confusion matrices of the readout components. `None` if there are no readout components | `readout_confusion({"readout": 0.03})` |

<br>

<!--
| 関数 | 引数 | 入力する値 | 省略時 |
|---|---|---|---|
| `build_noise_model`, `noise_components`, `readout_confusion` | `kind` | `NOISE_KINDS` の名前（`"gate"`, `"readout"`, `"both"` など）、または `{成分の名前: 強さ}` の辞書。強さは 0 以上で、成分の `max` 以下 | 必須 |
-->

| Function | Argument | Value to pass | If omitted |
|---|---|---|---|
| `build_noise_model`, `noise_components`, `readout_confusion` | `kind` | A name in `NOISE_KINDS` (`"gate"`, `"readout"`, `"both"`, and so on), or a dictionary `{component name: strength}`. Each strength must be at least 0 and at most the component's `max` | Required |

<br>

<!--
`NOISE_COMPONENTS` の各成分に書く項目は次のとおりです。

| 項目 | 内容 | 省略時 |
|---|---|---|
| `label` | 強さの意味。5.9 のグラフの横軸の名前にもなる | 必須 |
| `max` | 強さの上限。`None` なら上限なし | `None` |
| `gate_errors` | 強さ → `{(ゲートの名前, ...): QuantumError}` の関数。誤差をすべての量子ビットのそのゲートの後に加える | なし |
| `readout` | 強さ → 混同行列 `[[P(0|0), P(1|0)], [P(0|1), P(1|1)]]` の関数。すべての量子ビットの測定に加える | なし |
| `log_scale` | `True` なら、5.9 のグラフの横軸を、0〜1 は線形、1 以上は対数の目盛り（symlog）にする | `False` |
-->

The items written for each component in `NOISE_COMPONENTS` are as follows.

| Item | Contents | If omitted |
|---|---|---|
| `label` | What the strength means. Also used as the horizontal-axis label in 5.9 | Required |
| `max` | Upper limit of the strength. `None` means no upper limit | `None` |
| `gate_errors` | Function strength → `{(gate name, ...): QuantumError}`. The error is added after those gates on all qubits | None |
| `readout` | Function strength → confusion matrix `[[P(0|0), P(1|0)], [P(0|1), P(1|1)]]`. Added to the measurement of all qubits | None |
| `log_scale` | If `True`, the horizontal axis in 5.9 is linear from 0 to 1 and logarithmic above 1 (symlog) | `False` |

In [ ]:
# ---- Noise components ----
# Each component is one kind of noise whose strength can be set. To add a new kind of noise,
# add an entry here (the table in 5.2 lists the errors Qiskit Aer provides).
#   label       : what the strength means (also the horizontal-axis label in 5.9)
#   max         : largest allowed strength (None: no upper limit)
#   gate_errors : function strength -> {(gate names, ...): QuantumError}; the error is added after those gates on all qubits
#   readout     : function strength -> confusion matrix [[P(0|0), P(1|0)], [P(0|1), P(1|1)]] for all measurements
#   log_scale   : True to draw the horizontal axis in 5.9 on a symlog scale
NOISE_COMPONENTS = {
    "depolarizing": dict(
        label="Depolarizing parameter lambda", max=1,
        gate_errors=lambda s: {("rx", "ry"): depolarizing_error(s, 1)}),
    "thermal": dict(
        label="Thermal gate-time multiplier", max=None, log_scale=True,
        gate_errors=lambda s: {("rx", "ry"): thermal_relaxation_error(T1, T2, s * GATE_TIME)}),
    "readout": dict(
        label="Symmetric readout error epsilon", max=1,
        readout=lambda s: [[1 - s, s], [s, 1 - s]]),
}

# ---- Noise kinds used in 5.3-5.8: name -> {component: strength} ----
# Components are applied in the order written (here: depolarizing, then thermal relaxation).
# The strength of "thermal" is the multiplier of GATE_TIME (1 = the gate time in 5.1).
NOISE_KINDS = {
    "gate": {"depolarizing": DEPOLARIZING_PARAM, "thermal": 1},
    "readout": {"readout": READOUT_ERROR_RATE},
    "both": {"depolarizing": DEPOLARIZING_PARAM, "thermal": 1, "readout": READOUT_ERROR_RATE},
}


def noise_components(kind):
    # A name in NOISE_KINDS, or a dictionary {component: strength} -> the checked dictionary {component: strength}
    components = NOISE_KINDS.get(kind) if isinstance(kind, str) else kind
    if components is None:
        raise ValueError(f"kind must be one of {list(NOISE_KINDS)} or a dictionary {{component: strength}} (got: {kind!r})")
    for name, strength in components.items():
        if name not in NOISE_COMPONENTS:
            raise ValueError(f"Unknown noise component: {name!r} (defined: {list(NOISE_COMPONENTS)})")
        upper = NOISE_COMPONENTS[name].get("max")
        if not math.isfinite(strength) or strength < 0 or (upper is not None and strength > upper):
            raise ValueError(f"The strength of {name!r} must be between 0 and {upper} (got: {strength})")
    return components


def readout_confusion(kind):
    # Product of the confusion matrices of the readout components (None if there are none)
    confusion = None
    for name, strength in noise_components(kind).items():
        make = NOISE_COMPONENTS[name].get("readout")
        if make is None or strength == 0:
            continue
        matrix = np.array(make(strength), dtype=float)
        confusion = matrix if confusion is None else confusion @ matrix
    return confusion


def build_noise_model(kind):
    noise_model = NoiseModel(basis_gates=BASIS_GATES)
    # Errors on the same gate are composed in the order of the components and added once
    gate_errors = {}
    for name, strength in noise_components(kind).items():
        make = NOISE_COMPONENTS[name].get("gate_errors")
        if make is None or strength == 0:
            continue
        for gates, error in make(strength).items():
            for gate in gates:
                gate_errors[gate] = error if gate not in gate_errors else gate_errors[gate].compose(error)
    for gate, error in gate_errors.items():
        noise_model.add_all_qubit_quantum_error(error, [gate])
    # Readout errors are combined into one confusion matrix (adding a second ReadoutError would replace the first)
    confusion = readout_confusion(kind)
    if confusion is not None:
        noise_model.add_all_qubit_readout_error(ReadoutError(confusion))
    return noise_model


for kind in NOISE_KINDS:
    print(f"===== {kind} =====")
    print(build_noise_model(kind))

<!-- ### 5.3 ゲート・ノイズの種類・測定回数を選ぶ -->

### 5.3 Choosing the gate, noise type, and number of measurements

<!-- `GATE_NOISY` に `"H"`, `"Rx"`, `"Ry"` のいずれかを、`NOISE_KIND` に `"gate"`, `"readout"`, `"both"` のいずれか（または 5.2 の `NOISE_KINDS` に自分で加えた名前）を指定します。測定回数は $10^6$ にしています。この回数でも、ゲートノイズによる偏り（$R_X$, $R_Y$ で約 $10^{-5}$）は統計的な揺らぎ（約 $4 \times 10^{-4}$）に埋もれて検出できません。3σ で検出するには約 $2 \times 10^{10}$ shots が必要です。ここで確かめるのは、bias が揺らぎの範囲に収まることです。 -->

Set `GATE_NOISY` to one of `"H"`, `"Rx"`, `"Ry"`, and set `NOISE_KIND` to one of `"gate"`, `"readout"`, `"both"` (or a name you added to `NOISE_KINDS` in 5.2). The number of measurements is $10^6$. Even at this number, the bias from gate noise (about $10^{-5}$ for $R_X$ and $R_Y$) is buried in the statistical fluctuation (about $4 \times 10^{-4}$) and cannot be detected. Detecting it at 3σ would require about $2 \times 10^{10}$ shots. What we check here is that the bias stays within the range of the fluctuation.

In [ ]:
GATE_NOISY = "H"          # Choose from "H", "Rx", "Ry"
NOISE_KIND = "both"       # A name in NOISE_KINDS (5.2): "gate", "readout", "both", ...
SHOTS_NOISY = 1_000_000   # Number of measurements

noise_model = build_noise_model(NOISE_KIND)

<!-- ### 5.4 回路を作って transpile する -->

### 5.4 Building and transpiling the circuit

<!-- 4.2 と同じです。ノイズはゲート `rx`, `ry` に加えるので、transpile 後のゲートの数がそのまま「ノイズを受ける回数」になります（$H_{\mathrm{Gate}}$ は 2 回、$R_X$, $R_Y$ は 1 回）。 -->

This is the same as 4.2. Since noise is added at the `rx` and `ry` gates, the number of gates after transpilation is exactly the "number of times noise is received" (2 for $H_{\mathrm{Gate}}$; 1 for $R_X$ and $R_Y$).

In [ ]:
qc_noisy = build_c1(GATE_NOISY)
qc_noisy_t = transpile(qc_noisy, basis_gates=BASIS_GATES)

print(qc_noisy_t.draw())
print("Number of gates receiving noise:", sum(qc_noisy_t.count_ops().get(g, 0) for g in ("rx", "ry")))

<!-- ### 5.5 測定する -->

### 5.5 Measuring

<!-- 4.3 との違いは、`AerSimulator(noise_model=noise_model)` でノイズモデルを渡すところだけです。 -->

The only difference from 4.3 is that the noise model is passed in via `AerSimulator(noise_model=noise_model)`.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、このセルの raw dataを `5.5_noisy_single/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., the raw data of this cell is saved in `5.5_noisy_single/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
backend_noisy = AerSimulator(noise_model=noise_model, seed_simulator=SEED)

job = backend_noisy.run(qc_noisy_t, shots=SHOTS_NOISY, memory=True)
result = job.result()

counts_noisy = result.get_counts()
raw_bits_noisy = result.get_memory()

print("counts =", counts_noisy)
print("raw bits (first 50) =", "".join(raw_bits_noisy[:50]))
print("Length of raw bits =", len(raw_bits_noisy))

# Save the raw data (only when SAVE_RESULTS = True)
if SAVE_RESULTS:
    path = save_raw(output_dir("5.5_noisy_single") / f"c1_{GATE_NOISY}_noise-{NOISE_KIND}.txt", raw_bits_noisy,
                    GATE_NOISY, SHOTS_NOISY, SEED, noise_model, metadata={"condition": f"noise-{NOISE_KIND}"})
    print("Saved:", path)

<!-- ### 5.6 評価指標を計算する -->

### 5.6 Computing the evaluation metrics

<!-- 4.4 と同じ指標を計算します。説明パート E の予想どおり、bias が統計的な揺らぎの目安と同程度かどうかを確認してください。 -->

We compute the same metrics as in 4.4. Check whether, as predicted in Explanation Part E, the bias is comparable to the reference value for statistical fluctuation.

In [ ]:
r = evaluate(counts_noisy)
show_metrics(r)

<!-- ### 5.7 測定を繰り返す -->

### 5.7 Repeating the measurement

<!-- 4.5 の `run_trials` にノイズモデルを渡して、`N_TRIALS` 回繰り返します（$10^6$ shots × 10 回なので、少し時間がかかります）。 -->

We pass the noise model to `run_trials` from 4.5 and repeat `N_TRIALS` times ($10^6$ shots × 10 runs, so it takes a little while).

<!-- 1. で `SAVE_RESULTS = True` にした場合は、試行ごとのファイル（`c1_H_noise-both_trial001.txt` など）と `trials.csv`を `5.7_noisy_trials/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., one file per trial (`c1_H_noise-both_trial001.txt` and so on) and `trials.csv` are saved in `5.7_noisy_trials/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
rows_noisy = run_trials(GATE_NOISY, SHOTS_NOISY, N_TRIALS, noise_model=noise_model, seed=SEED,
                        raw_dir=output_dir("5.7_noisy_trials"), label=f"noise-{NOISE_KIND}")
show_table(rows_noisy, "trial")
summarize(rows_noisy)

<!-- ### 5.8 （参考）ゲートとノイズの種類をすべて比べる -->

### 5.8 (Optional) Comparing all gates and noise types

<!-- 3 つのゲート × 3 種類のノイズの 9 通りを 1 回ずつ測定して並べます（$10^6$ shots × 9 回）。 -->

We measure all 9 combinations of 3 gates × 3 noise types once each and list the results side by side ($10^6$ shots × 9 runs).

<!-- 1. で `SAVE_RESULTS = True` にした場合は、9 通りの raw data（`c1_H_noise-gate.txt` など）と一覧 `summary.csv`を `5.8_noisy_comparison/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., the raw data for the 9 combinations (`c1_H_noise-gate.txt` and so on) and their list, `summary.csv`, are saved in `5.8_noisy_comparison/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
conditions = [(gate, f"noise-{kind}", build_noise_model(kind)) for kind in NOISE_KINDS for gate in GATES]
rows = run_conditions(conditions, SHOTS_NOISY, "5.8_noisy_comparison")
show_table(rows, "gate / noise")
print(f"[Reference] expected bias for fair bits = {expected_bias(SHOTS_NOISY):.6f}")

<!-- ### 5.9 （参考）ノイズの大きさを段階的に変える -->

### 5.9 (Optional) Changing the noise strength step by step

<!-- 脱分極・熱緩和・読み出し誤差を**1 成分ずつ**変更し、ほかの成分をゼロにして原因を切り分けます。各条件で $H_{\mathrm{Gate}}$ / Rx / Ry を比較します。 -->

We vary the depolarizing, thermal relaxation, and readout errors **one component at a time**, setting the other components to zero to isolate the cause. Under each condition we compare $H_{\mathrm{Gate}}$ / Rx / Ry.

<!--
- 脱分極：`depolarizing_error` のパラメータ λ を 0〜0.10 で変えます（平均ゲート不忠実度そのものではありません）。
- 熱緩和：T1、T2 は 5.1 の値に固定し、ノイズに渡す有効ゲート時間を `倍率 × GATE_TIME` にします。倍率 0 は熱緩和なし、1 は元の条件です。大きい倍率は影響を観察するための仮想条件です。
- 読み出し：対称な読み出し誤差率 $\varepsilon$ を 0〜0.20 で変えます。
-->

- Depolarizing: vary the parameter λ of `depolarizing_error` from 0 to 0.10 (this is not the average gate infidelity itself).
- Thermal relaxation: fix T1 and T2 at the values in 5.1, and set the effective gate time passed to the noise to `multiplier × GATE_TIME`. A multiplier of 0 means no thermal relaxation, and 1 is the original condition. Large multipliers are hypothetical conditions for observing the effect.
- Readout: vary the symmetric readout error rate $\varepsilon$ from 0 to 0.20.

<!-- `SWEEP_VALUES` のリストを編集すると、範囲と刻み幅を自由に変更できます。100 万 shots × 3 試行で確認します。全条件数と総 shots は実行前に表示します。 -->

The range and step size can be changed freely by editing the lists in `SWEEP_VALUES`. We check with 1 million shots × 3 trials. The total number of conditions and total shots are displayed before running.

<!-- ノイズの成分は 5.2 の `NOISE_COMPONENTS` で定義しています。新しい成分を 5.2 で加えたあと、`SWEEP_VALUES` に成分の名前と強さのリストを加えると、その成分も強さを変えて測定します。グラフの列は成分の数だけ自動で増えます。 -->

The noise components are defined in `NOISE_COMPONENTS` in 5.2. After adding a new component in 5.2, add its name and a list of strengths to `SWEEP_VALUES` to vary its strength as well. The plot gets one column per component automatically.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `build_sweep_noise` | 指定した 1 成分だけを含む NoiseModel | `build_sweep_noise("thermal", 1000)` |
| `sweep_exact_p1` | 測定揺らぎを含まない、モデルから計算した P(1)（float） | `sweep_exact_p1("H", "thermal", 1000, build_sweep_noise("thermal", 1000))` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `build_sweep_noise` | NoiseModel containing only the specified single component | `build_sweep_noise("thermal", 1000)` |
| `sweep_exact_p1` | P(1) computed from the model, without measurement fluctuation (float) | `sweep_exact_p1("H", "thermal", 1000, build_sweep_noise("thermal", 1000))` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>build_sweep_noise</code></td><td><code>component</code></td><td>5.2 の <code>NOISE_COMPONENTS</code> にある成分の名前で、<code>SWEEP_VALUES</code> のキーにもあるもの（例：<code>"thermal"</code>）</td><td>必須</td></tr>
<tr><td><code>strength</code></td><td>有限の非負値。脱分極は λ（0〜1）、読み出しは誤り確率 ε（0〜1）、熱緩和は <code>GATE_TIME</code> の倍率（例：<code>1000</code>）。<code>0</code> はノイズなし</td><td>必須</td></tr>
<tr><td rowspan="4"><code>sweep_exact_p1</code></td><td><code>gate</code></td><td><code>"H"</code>・<code>"Rx"</code>・<code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td><code>component</code></td><td><code>build_sweep_noise</code> と同じノイズ成分名</td><td>必須</td></tr>
<tr><td><code>strength</code></td><td><code>build_sweep_noise</code> に渡したものと同じ強さ。読み出し誤差を P(1) に反映するときに使用</td><td>必須</td></tr>
<tr><td><code>nm</code></td><td>同じ <code>component</code>・<code>strength</code> で <code>build_sweep_noise</code> が返した NoiseModel</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>build_sweep_noise</code></td><td><code>component</code></td><td>A component name in <code>NOISE_COMPONENTS</code> (5.2) that is also a key of <code>SWEEP_VALUES</code> (e.g., <code>"thermal"</code>)</td><td>Required</td></tr>
<tr><td><code>strength</code></td><td>A finite non-negative value. λ (0–1) for depolarizing, error probability ε (0–1) for readout, multiplier of <code>GATE_TIME</code> (e.g., <code>1000</code>) for thermal relaxation. <code>0</code> means no noise</td><td>Required</td></tr>
<tr><td rowspan="4"><code>sweep_exact_p1</code></td><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td><code>component</code></td><td>Same noise component name as for <code>build_sweep_noise</code></td><td>Required</td></tr>
<tr><td><code>strength</code></td><td>Same strength as passed to <code>build_sweep_noise</code>. Used to reflect the readout error in P(1)</td><td>Required</td></tr>
<tr><td><code>nm</code></td><td>NoiseModel returned by <code>build_sweep_noise</code> for the same <code>component</code> and <code>strength</code></td><td>Required</td></tr>
</tbody>
</table>


In [ ]:
SWEEP_VALUES = {
    "depolarizing": [i / 100 for i in range(11)],
    "thermal": [0, 1, 10, 100, 250, 500, 1000, 2500, 5000],
    "readout": [i / 50 for i in range(11)],
}
SWEEP_GATES = list(GATES)
SWEEP_SHOTS = 1_000_000
SWEEP_TRIALS = 3
SWEEP_SEED = 2026


def build_sweep_noise(component, strength):
    # NoiseModel with only one component (all other components are zero)
    if component not in SWEEP_VALUES:
        raise ValueError(f"Unknown noise component: {component} (add it to SWEEP_VALUES)")
    return build_noise_model({component: strength})


def sweep_exact_p1(gate, component, strength, nm):
    # Save the density matrix before measurement. Readout error is applied with a classical confusion matrix.
    qc = transpile(build_c1(gate), basis_gates=BASIS_GATES, optimization_level=0)
    qc = qc.remove_final_measurements(inplace=False)
    qc.save_density_matrix()
    backend = AerSimulator(method="density_matrix", noise_model=nm)
    rho = backend.run(qc, shots=1).result().data(0)["density_matrix"]
    p1 = float(rho.data[1, 1].real)
    confusion = readout_confusion({component: strength})
    if confusion is not None:
        p1 = (1 - p1) * confusion[0, 1] + p1 * confusion[1, 1]
    return min(1.0, max(0.0, p1))

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `run_noise_sweep` | `(試行ごとの記録リスト, 条件ごとの要約リスト)` のタプル。`raw_dir` を指定すると、raw data と `trials.csv`・`summary.csv` をそのフォルダに保存する | `records, summary = run_noise_sweep(raw_dir="results/c1/2026-11-10_153012/5.9_noise_sweep")` |
| `measure_condition` | `(trials.csv の行のリスト, summary.csv の 1 行)`。1 つの条件で `SWEEP_TRIALS` 回測定し、密度行列からの値と平均・標準偏差をまとめる（`run_noise_sweep` の中で使う） | `measure_condition("thermal", 1000, "H", build_sweep_noise("thermal", 1000), None, 2026)` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `run_noise_sweep` | Tuple `(list of per-trial records, list of per-condition summaries)`. If `raw_dir` is given, the raw data and `trials.csv` and `summary.csv` are saved in that folder | `records, summary = run_noise_sweep(raw_dir="results/c1/2026-11-10_153012/5.9_noise_sweep")` |
| `measure_condition` | `(rows for trials.csv, one row for summary.csv)`. Measures `SWEEP_TRIALS` times under one condition and collects the density-matrix values, means, and standard deviations (used inside `run_noise_sweep`) | `measure_condition("thermal", 1000, "H", build_sweep_noise("thermal", 1000), None, 2026)` |

<br>

<!--
| 関数 | 引数 | 入力する値 | 省略時 |
|---|---|---|---|
| `run_noise_sweep` | `raw_dir` | 条件・試行別のビット列と JSON と、結果の CSV（`trials.csv`, `summary.csv`）の保存フォルダ。文字列または Path。`None` なら raw data も CSV も保存しない。空でないフォルダを指定すると測定前にエラーになるので、再実行時は未使用の保存先を指定 | `None` |
| `measure_condition` | `(trials.csv の行のリスト, summary.csv の 1 行)`。1 つの条件で `SWEEP_TRIALS` 回測定し、密度行列からの値と平均・標準偏差をまとめる（`run_noise_sweep` の中で使う） | `measure_condition("thermal", 1000, "H", build_sweep_noise("thermal", 1000), None, 2026)` |
| `measure_condition` | `(trials.csv の行のリスト, summary.csv の 1 行)`。1 つの条件で `SWEEP_TRIALS` 回測定し、密度行列からの値と平均・標準偏差をまとめる（`run_noise_sweep` の中で使う） | `measure_condition("thermal", 1000, "H", build_sweep_noise("thermal", 1000), None, 2026)` |
| `measure_condition` | `(trials.csv の行のリスト, summary.csv の 1 行)`。1 つの条件で `SWEEP_TRIALS` 回測定し、密度行列からの値と平均・標準偏差をまとめる（`run_noise_sweep` の中で使う） | `measure_condition("thermal", 1000, "H", build_sweep_noise("thermal", 1000), None, 2026)` |
-->

| Function | Argument | Value to pass | If omitted |
|---|---|---|---|
| `run_noise_sweep` | `raw_dir` | Folder in which to save the bit strings and JSON for each condition and trial, and the CSV files of the results (`trials.csv`, `summary.csv`), as a string or Path. `None` means neither the raw data nor the CSV files are saved. A non-empty folder raises an error before measuring, so specify an unused destination when re-running | `None` |
| `measure_condition` | `(rows for trials.csv, one row for summary.csv)`. Measures `SWEEP_TRIALS` times under one condition and collects the density-matrix values, means, and standard deviations (used inside `run_noise_sweep`) | `measure_condition("thermal", 1000, "H", build_sweep_noise("thermal", 1000), None, 2026)` |
| `measure_condition` | `(rows for trials.csv, one row for summary.csv)`. Measures `SWEEP_TRIALS` times under one condition and collects the density-matrix values, means, and standard deviations (used inside `run_noise_sweep`) | `measure_condition("thermal", 1000, "H", build_sweep_noise("thermal", 1000), None, 2026)` |
| `measure_condition` | `(rows for trials.csv, one row for summary.csv)`. Measures `SWEEP_TRIALS` times under one condition and collects the density-matrix values, means, and standard deviations (used inside `run_noise_sweep`) | `measure_condition("thermal", 1000, "H", build_sweep_noise("thermal", 1000), None, 2026)` |

<br>

<!-- 引数以外の実験条件は直前の設定セルで変更します。 -->

Experimental conditions other than the arguments are changed in the preceding settings cell.

<!--
| 設定変数 | 入力する値 |
|---|---|
| `SWEEP_VALUES` | 成分名（5.2 の `NOISE_COMPONENTS` の名前）をキー、試す強さのリストを値とする辞書 |
| `SWEEP_GATES` | 比較するゲートのリスト（例：`["H", "Rx", "Ry"]`） |
| `SWEEP_SHOTS` | 1 試行あたりの測定回数。1 以上の整数 |
| `SWEEP_TRIALS` | 条件ごとの試行数。標準偏差の計算のため 2 以上の整数 |
| `SWEEP_SEED` | 非負整数または `None` |
| `SWEEP_RAW_DIR` | 次のコードセル末尾の呼び出しに渡す保存先。1. で `SAVE_SWEEP = True` なら `RUN_DIR/5.9_noise_sweep`、`False` なら `None` に自動で決まる |
-->

| Setting variable | Value to set |
|---|---|
| `SWEEP_VALUES` | Dictionary with component names (names in `NOISE_COMPONENTS` in 5.2) as keys and lists of strengths to try as values |
| `SWEEP_GATES` | List of gates to compare (e.g., `["H", "Rx", "Ry"]`) |
| `SWEEP_SHOTS` | Number of measurements per trial. An integer of at least 1 |
| `SWEEP_TRIALS` | Number of trials per condition. An integer of at least 2, for computing the standard deviation |
| `SWEEP_SEED` | Non-negative integer or `None` |
| `SWEEP_RAW_DIR` | Destination passed to the call at the end of the next code cell. Set automatically: `RUN_DIR/5.9_noise_sweep` if `SAVE_SWEEP = True` in 1., otherwise `None` |


In [ ]:
SWEEP_METRICS = ("P1", "bias", "shannon", "min_entropy")


def measure_condition(component, strength, gate, nm, raw_dir, first_seed):
    # SWEEP_TRIALS measurements under one condition -> (rows for trials.csv, one row for summary.csv)
    rows = []
    for trial in range(1, SWEEP_TRIALS + 1):
        name = f"c1_{gate}_{component}-{strength:g}_trial{trial:03d}.txt"
        raw_path = None if raw_dir is None else Path(raw_dir) / component / name
        seed = offset_seed(first_seed, trial - 1)
        r = run_c1(gate, SWEEP_SHOTS, nm, seed, raw_path, method="density_matrix", optimization_level=0,
                   metadata={"component": component, "strength": strength, "trial": trial})
        rows.append(dict(component=component, strength=strength, gate=gate, trial=trial, seed=seed, **r))
    p1 = sweep_exact_p1(gate, component, strength, nm)
    summary = dict(component=component, strength=strength, gate=gate,
                   exact_P1=p1, exact_bias=abs(p1 - 0.5), exact_min_entropy=min_entropy(1 - p1, p1))
    for metric in SWEEP_METRICS:
        values = [row[metric] for row in rows]
        summary[metric + "_mean"] = statistics.mean(values)
        summary[metric + "_std"] = statistics.stdev(values)
    return rows, summary


def run_noise_sweep(raw_dir=None):
    if SWEEP_SHOTS < 1 or SWEEP_TRIALS < 2:
        raise ValueError("shots must be at least 1, and the number of trials must be at least 2 to compute the standard deviation")
    if raw_dir is not None and Path(raw_dir).exists() and any(Path(raw_dir).iterdir()):
        raise FileExistsError(f"Destination is not empty: {raw_dir}. Run the Setup cell (1.) again to save in a new folder.")
    conditions = len(SWEEP_GATES) * sum(map(len, SWEEP_VALUES.values()))
    print(f"{conditions} conditions × {SWEEP_TRIALS} trials × {SWEEP_SHOTS:,} shots"
          f" = {conditions * SWEEP_TRIALS * SWEEP_SHOTS:,} shots")
    records, summary = [], []
    for component, strengths in SWEEP_VALUES.items():
        for strength in strengths:
            nm = build_sweep_noise(component, strength)
            for gate in SWEEP_GATES:
                first_seed = offset_seed(SWEEP_SEED, len(records))
                rows, item = measure_condition(component, strength, gate, nm, raw_dir, first_seed)
                records += rows
                summary.append(item)
        print(f"Done: {component}")
    if raw_dir is not None:
        # The CSV files are saved only together with the raw data, after all conditions have been measured
        for name, data in [("trials.csv", records), ("summary.csv", summary)]:
            print("Saved:", write_csv(Path(raw_dir) / name, data))
    return records, summary


# Saved in RUN_DIR/5.9_noise_sweep only when SAVE_SWEEP = True in 1. Setup (None: not saved)
SWEEP_RAW_DIR = RUN_DIR / "5.9_noise_sweep" if SAVE_SWEEP else None
sweep_records, sweep_summary = run_noise_sweep(raw_dir=SWEEP_RAW_DIR)

In [ ]:
# Points and error bars: trial mean ± between-trial standard deviation (clipped to the range the metric can take)
# Solid lines: values computed from the density matrix
PLOT_ROWS = [("P1", "P(1)", (0, 1)),
             ("bias", "Bias", (0, 0.5)),
             ("min_entropy", "Min-entropy", (0, 1))]


def plot_gate(ax_col, component, gate, color):
    # One gate in one column: sampled points with error bars and the exact line, in each row
    points = sorted((r for r in sweep_summary if r["component"] == component and r["gate"] == gate),
                    key=lambda r: r["strength"])
    xs = [r["strength"] for r in points]
    for ax, (metric, ylabel, (low, high)) in zip(ax_col, PLOT_ROWS):
        means = [r[metric + "_mean"] for r in points]
        stds = [r[metric + "_std"] for r in points]
        yerr = [[min(s, m - low) for m, s in zip(means, stds)],
                [min(s, high - m) for m, s in zip(means, stds)]]
        ax.errorbar(xs, means, yerr=yerr, fmt="o", capsize=3, color=color, label=gate + " sampled")
        ax.plot(xs, [r["exact_" + metric] for r in points], color=color, label=gate + " exact")
        ax.set(xlabel=NOISE_COMPONENTS[component]["label"], ylabel=ylabel)
        ax.grid(alpha=0.3)


n_cols = len(SWEEP_VALUES)   # one column per noise component
fig, axes = plt.subplots(len(PLOT_ROWS), n_cols, figsize=(5 * n_cols, 11), squeeze=False)
for col, component in enumerate(SWEEP_VALUES):
    ax_col = axes[:, col]
    for gate_index, gate in enumerate(SWEEP_GATES):
        plot_gate(ax_col, component, gate, f"C{gate_index}")
    ax_col[0].axhline(0.5, color="gray", linestyle=":")
    fair_line = ax_col[1].axhline(expected_bias(SWEEP_SHOTS), color="gray", linestyle=":",
                                  label="Fair-bit expected sampled bias")
    ax_col[0].legend(fontsize=8)
    ax_col[1].legend(handles=[fair_line], fontsize=8)
    if NOISE_COMPONENTS[component].get("log_scale"):
        # Multiplier 0 is included, so use a linear scale for 0–1 and a logarithmic scale above 1
        for ax in ax_col:
            ax.set_xscale("symlog", linthresh=1)
            ax.set_xlim(-0.3, max(SWEEP_VALUES[component]) * 1.5)
fig.tight_layout()
plt.show()

lines = ["| noise | strength | gate | P(1) mean ± SD | exact P(1) | bias mean | min-entropy mean |",
         "|---|---|---|---|---|---|---|"]
for r in sweep_summary:
    lines.append(f"| {r['component']} | {r['strength']:g} | {r['gate']} | "
                 f"{r['P1_mean']:.6f} ± {r['P1_std']:.6f} | {r['exact_P1']:.6f} | "
                 f"{r['bias_mean']:.6f} | {r['min_entropy_mean']:.6f} |")
display(Markdown("\n".join(lines)))

<!-- #### 結果の読み方と保存 -->

#### How to read the results and save them

<!-- **グラフは 3 行 × 3 列です。まず中央列の上段から見ると、熱緩和による偏りを確認できます。** -->

**The figure has 3 rows × 3 columns. Start with the top row of the center column to see the bias caused by thermal relaxation.**

<!--
| 段（縦軸） | 左列：脱分極 | 中央列：熱緩和 | 右列：読み出し誤差 | 値の意味 |
|---|---|---|---|---|
| 上段：P(1) | 脱分極の P(1) | 熱緩和の P(1) | 読み出し誤差の P(1) | 1 が出る確率。偏りがないときは 0.5 |
| 中段：Bias | 脱分極の bias | 熱緩和の bias | 読み出し誤差の bias | P(1) と 0.5 の差の絶対値。0 に近いほど偏りが小さい |
| 下段：Min-entropy | 脱分極の min-entropy | 熱緩和の min-entropy | 読み出し誤差の min-entropy | −log₂(max(P(0), P(1)))。1 に近いほど測定分布の偏りが小さい |
-->

| Row (vertical axis) | Left column: depolarizing | Center column: thermal relaxation | Right column: readout error | Meaning of the value |
|---|---|---|---|---|
| Top: P(1) | P(1) for depolarizing | P(1) for thermal relaxation | P(1) for readout error | Probability of getting 1. 0.5 when there is no bias |
| Middle: Bias | Bias for depolarizing | Bias for thermal relaxation | Bias for readout error | Absolute difference between P(1) and 0.5. The closer to 0, the smaller the bias |
| Bottom: Min-entropy | Min-entropy for depolarizing | Min-entropy for thermal relaxation | Min-entropy for readout error | −log₂(max(P(0), P(1))). The closer to 1, the less biased the measurement distribution |

<!-- **横軸は、各ノイズの大きさを指定する値です。** 同じ列では、上・中・下段の横軸は共通です。 -->

**The horizontal axis is the value that sets the strength of each noise.** Within a column, the top, middle, and bottom rows share the same horizontal axis.

<!--
| 列 | 横軸の英語表記 | 横軸の意味と例 |
|---|---|---|
| 左：脱分極 | Depolarizing parameter lambda | 脱分極パラメータ λ。0 は脱分極なし、0.10 は状態を 10% の割合で最大混合状態へ近づける設定。平均ゲート不忠実度そのものではありません |
| 中央：熱緩和 | Thermal gate-time multiplier | 熱緩和の計算に使うゲート時間の倍率。0 は熱緩和なし、1 は元の 20 ns、1000 は 20,000 ns。T1 と T2 は固定しています。小さい倍率も見分けられるよう、0〜1 は線形、1 以上は対数の目盛（symlog）で表示しています |
| 右：読み出し誤差 | Symmetric readout error epsilon | 0→1 と 1→0 を読み間違える確率 $\varepsilon$。0 は誤りなし、0.10 はそれぞれ 10% の確率で読み間違える設定 |
-->

| Column | Horizontal-axis label | Meaning of the horizontal axis and examples |
|---|---|---|
| Left: depolarizing | Depolarizing parameter lambda | Depolarizing parameter λ. 0 means no depolarizing; 0.10 means the state is moved toward the maximally mixed state with weight 10%. This is not the average gate infidelity itself |
| Center: thermal relaxation | Thermal gate-time multiplier | Multiplier on the gate time used in the thermal relaxation calculation. 0 means no thermal relaxation, 1 is the original 20 ns, and 1000 is 20,000 ns. T1 and T2 are held fixed. So that small multipliers can also be distinguished, the scale is linear from 0 to 1 and logarithmic above 1 (symlog) |
| Right: readout error | Symmetric readout error epsilon | Probability $\varepsilon$ of misreading 0→1 and 1→0. 0 means no errors; 0.10 means each is misread with 10% probability |

<!-- 今回の横軸の範囲では、右へ進むほど各ノイズを強くしています。列ごとに単位と意味が違うので、列をまたいで同じ数値を同じノイズ量とは解釈しません。 -->

In the ranges used here, each noise gets stronger toward the right. Because the units and meanings differ from column to column, the same number in different columns should not be interpreted as the same amount of noise.

<!-- **点・実線・エラーバーの見方** -->

**How to read the points, solid lines, and error bars**

<!--
- 色は H・Rx・Ry の違いを表します（凡例や表の H は $H_{\mathrm{Gate}}$ です）。同じ色の点と実線を見比べてください。結果がほぼ同じ場合、線や点が重なって見えます。
- 点（`sampled`）は、有限 shots の測定を繰り返して得た指標の平均です。
- 点の上下の棒（エラーバー）は、試行間の標準偏差です。棒が長いほど試行ごとのばらつきが大きいことを示します。信頼区間ではありません。指標が取り得る範囲（bias は 0 以上、min-entropy は 1 以下など）で切って表示しているので、切った側の棒は標準偏差より短くなります。CSV の標準偏差は元の値のままです。
- 実線（`exact`）は、設定したノイズモデルから密度行列を使って求めた P(1)、およびその確率から計算した bias・min-entropy です。読み出し誤差は混同行列で反映しています。有限 shots による統計的な揺らぎを含まず、測定を非常に多く繰り返したときに近づく値を示します。実機で必ず同じ値になるという意味ではありません。
- 上段の灰色の点線は P(1) = 0.5、中段の灰色の点線は公平なビットを `SWEEP_SHOTS` 回測定した場合の bias の期待値の目安です。
-->

- Colors distinguish H, Rx, and Ry (H in the legend and tables is $H_{\mathrm{Gate}}$). Compare points and solid lines of the same color. When results are nearly identical, the lines and points overlap.
- The points (`sampled`) are the average of the metric obtained by repeating measurements with a finite number of shots.
- The bars above and below the points (error bars) are the standard deviation between trials. Longer bars indicate larger trial-to-trial variation. They are not confidence intervals. Because they are clipped to the range the metric can take (bias at least 0, min-entropy at most 1, and so on), a clipped bar is shorter than the standard deviation on that side. The standard deviations in the CSV are the unclipped values.
- The solid lines (`exact`) are P(1) obtained from the configured noise model using density matrices, and the bias and min-entropy computed from that probability. Readout error is included through the confusion matrix. They contain no statistical fluctuation from finite shots and show the values approached when measurements are repeated a very large number of times. This does not mean real hardware will necessarily give the same values.
- The gray dotted line in the top row is P(1) = 0.5, and the gray dotted line in the middle row is the reference expected bias when a fair bit is measured `SWEEP_SHOTS` times.

<!-- bias と min-entropy は各試行で求めてから平均します。そのため、平均 P(1) から計算した値とは一般に一致しません。ノイズがゼロでも有限 shots の bias はゼロにはならず、min-entropy も 1 より小さく見えます。 -->

The bias and min-entropy are computed for each trial and then averaged. They therefore do not in general match the values computed from the average P(1). Even with zero noise, the finite-shot bias is not zero, and the min-entropy appears smaller than 1.

<!-- **結果を読む順序** -->

**Order in which to read the results**

<!--
1. 中央列の上段で、横軸を右へ進めたときに P(1) の実線が 0.5 から離れるかを見ます。熱緩和は |1⟩ を |0⟩ へ移すため、今回の設定では P(1) が下がります。
2. 同じ列の中段・下段を見ます。P(1) の偏りが大きくなると、bias は増え、min-entropy は下がります。
3. 点と実線を比べ、測定のばらつきに対して変化がどの程度大きいかを確認します。元の熱緩和条件（倍率 1）の影響は非常に小さいので、表の `exact P(1)` も確認してください。
4. 左列と右列を見ます。C1 の測定では、脱分極や対称な読み出し誤差を大きくしても P(1) がほぼ 1/2 のままになることがあります。水平な線は、ノイズが測定分布の偏りに現れていないことを示し、ノイズがないことを意味しません。
-->

1. In the top row of the center column, see whether the solid P(1) line moves away from 0.5 as you go right along the horizontal axis. Since thermal relaxation moves |1⟩ to |0⟩, P(1) decreases in this setup.
2. Look at the middle and bottom rows of the same column. As the bias in P(1) grows, the bias increases and the min-entropy decreases.
3. Compare the points and solid lines to see how large the change is relative to the measurement spread. The effect of the original thermal relaxation condition (multiplier 1) is extremely small, so also check `exact P(1)` in the table.
4. Look at the left and right columns. In the C1 measurement, P(1) can remain nearly 1/2 even when the depolarizing or symmetric readout error is increased. A horizontal line indicates that the noise does not show up as a bias in the measurement distribution; it does not mean that there is no noise.

<!-- **$H_{\mathrm{Gate}}$ と Rx・Ry の熱緩和による変化が異なる理由** -->

**Why $H_{\mathrm{Gate}}$ and Rx/Ry change differently under thermal relaxation**

<!-- 今回の変換後の回路では、$H_{\mathrm{Gate}}$ は Ry(π/2) → Rx(π) の 2 個のゲートで実行され、各ゲートの後にノイズが入ります。最初の熱緩和で 0 が多くなっても、続く Rx(π) が 0 と 1 の確率を入れ替えるため、最後の熱緩和による偏りが一部相殺されます。Rx・Ry は 1 個のゲートの後に熱緩和が入ります。この順序と回数の違いが、$H_{\mathrm{Gate}}$ の P(1) が他より 0.5 に近くなる理由です。これは今回の分解方法とノイズモデルでの結果であり、$H_{\mathrm{Gate}}$ が常にノイズに強いという意味ではありません。 -->

In the transpiled circuit used here, $H_{\mathrm{Gate}}$ is executed as the two gates Ry(π/2) → Rx(π), with noise inserted after each gate. Even if the first thermal relaxation increases the number of 0s, the subsequent Rx(π) swaps the probabilities of 0 and 1, so the bias partly cancels against that from the final thermal relaxation. For Rx and Ry, thermal relaxation occurs after a single gate. This difference in order and count is why P(1) for $H_{\mathrm{Gate}}$ is closer to 0.5 than the others. This is a result for this particular decomposition and noise model; it does not mean that $H_{\mathrm{Gate}}$ is always more robust to noise.

<!-- 1 ゲートの間に |1⟩ が |0⟩ に落ちる確率を $\gamma$ とすると、bias は Rx・Ry で $\gamma/2$、$H_{\mathrm{Gate}}$ で $\gamma^2/2$ です。 -->

If $\gamma$ is the probability that |1⟩ decays to |0⟩ during one gate, the bias is $\gamma/2$ for Rx and Ry and $\gamma^2/2$ for $H_{\mathrm{Gate}}$.

<!-- 一方、論文の Odra 5 の実機では、C2 を除く回路群で $H_{\mathrm{Gate}}$ の平均が最も 50% から離れていました。論文は、$H_{\mathrm{Gate}}$ が複数のネイティブパルスから作られるため、小さな校正誤差が増幅されうることと整合すると述べています。5.9 のモデルには校正誤差を入れていないので、この効果は現れません。 -->

On the other hand, on the real Odra 5 hardware in the paper, the average for $H_{\mathrm{Gate}}$ was the farthest from 50% across the group of circuits excluding C2. The paper says this is consistent with small calibration errors being amplified, since $H_{\mathrm{Gate}}$ is built from multiple native pulses. The model in 5.9 includes no calibration errors, so this effect does not appear.

<!-- ここでの min-entropy は測定分布からの値であり、量子的な乱数の安全性を保証する値ではありません。 -->

The min-entropy here is a value derived from the measurement distribution and does not guarantee the security of the quantum random numbers.

<!-- **結果と raw data の保存** -->

**Saving the results and the raw data**

<!-- 1. で `SAVE_SWEEP = True` にした場合、`run_noise_sweep` は測定時に各条件・試行のビット列（`.txt`）と条件・集計結果（`.txt.json`）を `RUN_DIR/5.9_noise_sweep/<ノイズの種類>/` に保存し、すべての条件の測定が終わったあとに、`5.9_noise_sweep/` に次の CSV を保存します。 -->

If `SAVE_SWEEP = True` in 1., `run_noise_sweep` saves the bit string (`.txt`) and the conditions and aggregated results (`.txt.json`) for each condition and trial in `RUN_DIR/5.9_noise_sweep/<noise type>/` during measurement, and after all conditions have been measured, it saves the following CSV files in `5.9_noise_sweep/`.

<!--
- `trials.csv`：試行ごとの全結果。
- `summary.csv`：条件ごとの平均・標準偏差・密度行列による値。
-->

- `trials.csv`: all results for each trial.
- `summary.csv`: per-condition means, standard deviations, and density-matrix values.

<!--
- `SAVE_SWEEP = False`（既定）の場合は、raw data も CSV も保存しません。
- CSV は、同じ実行の raw data と同じフォルダにだけ保存します。CSV だけが新しい結果に書き換わって、raw data と対応しなくなることはありません。
- 同じ `RUN_DIR` のまま測定のセルをもう一度実行すると、保存先が空でないので、測定を始める前にエラーになります。もう一度保存するときは、1. のセルを実行し直してください。
- 測定が途中で止まった場合、そのフォルダには CSV が作られません。CSV がないフォルダは、途中までの raw data です。
- CSV のcountsから測定順のビット列は復元できないため、raw data が必要なら測定前に `SAVE_SWEEP = True` にしてください。
- ビット列の取得・保存には追加のメモリとディスク容量を使います（テキスト本体は 1 shot あたり 1 byte）。
-->

- With `SAVE_SWEEP = False` (the default), neither the raw data nor the CSV files are saved.
- The CSV files are saved only in the same folder as the raw data of the same run. The CSV files are never replaced by newer results that no longer match the raw data.
- If you run the measurement cell again with the same `RUN_DIR`, the destination is not empty, so an error is raised before any measurement. To save again, rerun the cell in 1.
- If the measurement stops partway, no CSV files are created in that folder. A folder without CSV files holds the raw data of an incomplete run.
- The bit strings in measurement order cannot be recovered from the counts in the CSV, so if you need raw data, set `SAVE_SWEEP = True` before measuring.
- Obtaining and saving the bit strings uses additional memory and disk space (the text itself takes 1 byte per shot).

<!-- 下のセルは、保存した CSV の場所を表示します。 -->

The cell below shows where the CSV files were saved.

In [ ]:
# The CSV files are saved by run_noise_sweep in the cell above, in SWEEP_RAW_DIR together with the raw data.
if SWEEP_RAW_DIR is None:
    print("SAVE_SWEEP = False, so neither the raw data nor the CSV files were saved.")
else:
    for filename in ("trials.csv", "summary.csv"):
        path = Path(SWEEP_RAW_DIR) / filename
        print(("Saved: " if path.is_file() else "Not saved: ") + str(path))